<a href="https://colab.research.google.com/github/acorrea79/techchallenge-fase2-pipeline-alfabetizacao/blob/main/notebooks/pipeline_alfabetizacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TechChallenge Fase 2 - Pipeline Híbrido para Análise da Alfabetização no Brasil

Projeto desenvolvido para a FIAP, turma 1IAST.

Integrante:

- Andre Correa Luis Vilas Boas

## Objetivo do notebook

Este notebook centraliza a execução acadêmica da pipeline de dados para análise da alfabetização no Brasil.

A solução utiliza:

- GCP BigQuery Sandbox;
- Google Colab;
- Python;
- Pandas;
- arquitetura Medalhão com camadas Bronze, Silver e Gold;
- ingestão Batch;
- simulação de Streaming;
- validações de qualidade;
- logs de monitoramento.

Este notebook faz parte da entrega do TechChallenge Fase 2.

## Instalação das bibliotecas

In [1]:
!pip install -q pandas numpy pyarrow google-cloud-bigquery db-dtypes

## Imports básicos do projeto

In [2]:
import os
import json
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
import numpy as np

from google.colab import auth
from google.cloud import bigquery

## Configurações do projeto

In [3]:
# Configurações principais do projeto

PROJECT_ID = "fiap-techchallenge-fase2"
DATASET_ID = "basedosdados.br_inep_avaliacao_alfabetizacao"

BASE_PATH = Path("/content/techchallenge-fase2-pipeline-alfabetizacao")

BRONZE_PATH = BASE_PATH / "data" / "bronze"
BRONZE_BATCH_PATH = BRONZE_PATH / "batch"
BRONZE_STREAMING_PATH = BRONZE_PATH / "streaming"

SILVER_PATH = BASE_PATH / "data" / "silver"
GOLD_PATH = BASE_PATH / "data" / "gold"
LOGS_PATH = BASE_PATH / "logs"

TABLES = {
    "alunos": f"{DATASET_ID}.alunos",
    "dicionario": f"{DATASET_ID}.dicionario",
    "meta_alfabetizacao_brasil": f"{DATASET_ID}.meta_alfabetizacao_brasil",
    "meta_alfabetizacao_uf": f"{DATASET_ID}.meta_alfabetizacao_uf",
    "meta_alfabetizacao_municipio": f"{DATASET_ID}.meta_alfabetizacao_municipio",
    "municipio": f"{DATASET_ID}.municipio",
    "uf": f"{DATASET_ID}.uf",
}

print("Configuração carregada com sucesso.")
print(f"Projeto GCP: {PROJECT_ID}")
print(f"Dataset principal: {DATASET_ID}")

Configuração carregada com sucesso.
Projeto GCP: fiap-techchallenge-fase2
Dataset principal: basedosdados.br_inep_avaliacao_alfabetizacao


## Criação das pastas da pipeline

In [4]:
# Criação da estrutura de pastas no ambiente Colab

paths = [
    BRONZE_BATCH_PATH,
    BRONZE_STREAMING_PATH,
    SILVER_PATH,
    GOLD_PATH,
    LOGS_PATH,
]

for path in paths:
    path.mkdir(parents=True, exist_ok=True)

print("Estrutura de pastas criada no Colab:")
for path in paths:
    print(f"- {path}")

Estrutura de pastas criada no Colab:
- /content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/batch
- /content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/streaming
- /content/techchallenge-fase2-pipeline-alfabetizacao/data/silver
- /content/techchallenge-fase2-pipeline-alfabetizacao/data/gold
- /content/techchallenge-fase2-pipeline-alfabetizacao/logs


## Função simples de log

In [5]:
# Função de log para monitoramento simples da pipeline

def write_log(message: str, level: str = "INFO"):
    LOGS_PATH.mkdir(parents=True, exist_ok=True)

    log_file = LOGS_PATH / "pipeline_execution.log"
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    log_line = f"{timestamp} | {level} | {message}"

    with open(log_file, "a", encoding="utf-8") as file:
        file.write(log_line + "\n")

    print(log_line)


write_log("Notebook principal iniciado.")

2026-07-12 20:59:40 | INFO | Notebook principal iniciado.


## Autenticação no Google

In [6]:
# Autenticação no Google
# Será aberta uma janela para autorizar o acesso com sua conta Google.

auth.authenticate_user()

write_log("Autenticação no Google realizada com sucesso.")

2026-07-12 21:01:12 | INFO | Autenticação no Google realizada com sucesso.


## Criação do cliente BigQuery

In [7]:
# Criação do cliente BigQuery

client = bigquery.Client(project=PROJECT_ID)

write_log("Cliente BigQuery criado com sucesso.")

2026-07-12 21:01:12 | INFO | Cliente BigQuery criado com sucesso.


##Teste de acesso ao BigQuery

In [8]:
# Teste simples de acesso ao BigQuery Sandbox

query = """
SELECT
  'BigQuery Sandbox acessado com sucesso' AS status
"""

df_test = client.query(query).to_dataframe(create_bqstorage_client=False)

display(df_test)

write_log("Consulta de teste executada com sucesso.")

,status
0,BigQuery Sandbox acessado com sucesso


2026-07-12 21:01:14 | INFO | Consulta de teste executada com sucesso.


# Listar tabelas do dataset principal

In [9]:
!gcloud auth list
!gcloud config set project fiap-techchallenge-fase2
!gcloud config get-value project

query_tables = """
SELECT
  table_catalog,
  table_schema,
  table_name,
  table_type
FROM
  `basedosdados.br_inep_avaliacao_alfabetizacao.INFORMATION_SCHEMA.TABLES`
ORDER BY
  table_name
"""

df_tables = client.query(query_tables).to_dataframe(create_bqstorage_client=False)

display(df_tables)

write_log(f"Listagem de tabelas executada. Total de tabelas encontradas: {len(df_tables)}")

      Credentialed Accounts
ACTIVE  ACCOUNT
*       gabrielhcorrea27@gmail.com

To set the active account, run:
    $ gcloud config set account `ACCOUNT`



To take a quick anonymous survey, run:
  $ gcloud survey

Updated property [core/project].
fiap-techchallenge-fase2


,table_catalog,table_schema,table_name,table_type
0,basedosdados,br_inep_avaliacao_alfabetizacao,alunos,BASE TABLE
1,basedosdados,br_inep_avaliacao_alfabetizacao,dicionario,BASE TABLE
2,basedosdados,br_inep_avaliacao_alfabetizacao,meta_alfabetizacao_brasil,BASE TABLE
3,basedosdados,br_inep_avaliacao_alfabetizacao,meta_alfabetizacao_municipio,BASE TABLE
4,basedosdados,br_inep_avaliacao_alfabetizacao,meta_alfabetizacao_uf,BASE TABLE
5,basedosdados,br_inep_avaliacao_alfabetizacao,municipio,BASE TABLE
6,basedosdados,br_inep_avaliacao_alfabetizacao,uf,BASE TABLE


2026-07-12 21:01:19 | INFO | Listagem de tabelas executada. Total de tabelas encontradas: 7


## Listar colunas do dataset principal

In [10]:
# Listagem das colunas das tabelas do dataset principal

query_columns = """
SELECT
  table_name,
  column_name,
  data_type,
  is_nullable,
  ordinal_position
FROM
  `basedosdados.br_inep_avaliacao_alfabetizacao.INFORMATION_SCHEMA.COLUMNS`
ORDER BY
  table_name,
  ordinal_position
"""

df_columns = client.query(query_columns).to_dataframe(create_bqstorage_client=False)

display(df_columns)

write_log(f"Schema carregado com sucesso. Total de colunas encontradas: {len(df_columns)}")

,table_name,column_name,data_type,is_nullable,ordinal_position
0,alunos,ano,INT64,YES,1
1,alunos,id_municipio,STRING,YES,2
2,alunos,id_escola,STRING,YES,3
3,alunos,id_aluno,STRING,YES,4
4,alunos,caderno,STRING,YES,5
...,...,...,...,...,...
78,uf,proporcao_aluno_nivel_4,FLOAT64,YES,11
79,uf,proporcao_aluno_nivel_5,FLOAT64,YES,12
80,uf,proporcao_aluno_nivel_6,FLOAT64,YES,13
81,uf,proporcao_aluno_nivel_7,FLOAT64,YES,14


2026-07-12 21:01:20 | INFO | Schema carregado com sucesso. Total de colunas encontradas: 83


## Volume das tabelas

In [11]:
# Consulta de volume das tabelas
# Esta etapa apoia decisões de FinOps.
# Tentamos primeiro INFORMATION_SCHEMA.TABLE_STORAGE.
# Caso falhe por restrição de localização/metadados, usamos __TABLES__ como alternativa.

query_storage_information_schema = """
SELECT
  table_name,
  total_rows,
  ROUND(total_logical_bytes / 1024 / 1024, 2) AS tamanho_mb
FROM
  `basedosdados.br_inep_avaliacao_alfabetizacao.INFORMATION_SCHEMA.TABLE_STORAGE`
ORDER BY
  total_logical_bytes DESC
"""

query_storage_legacy = """
SELECT
  table_id AS table_name,
  row_count AS total_rows,
  ROUND(size_bytes / 1024 / 1024, 2) AS tamanho_mb
FROM
  `basedosdados.br_inep_avaliacao_alfabetizacao.__TABLES__`
ORDER BY
  size_bytes DESC
"""

try:
    df_storage = client.query(query_storage_information_schema).to_dataframe(
        create_bqstorage_client=False
    )
    storage_source = "INFORMATION_SCHEMA.TABLE_STORAGE"
    write_log("Consulta de volume executada via INFORMATION_SCHEMA.TABLE_STORAGE.")

except Exception as error:
    write_log(
        f"Consulta via INFORMATION_SCHEMA.TABLE_STORAGE falhou. "
        f"Usando alternativa __TABLES__. Erro: {str(error)[:200]}",
        level="WARNING"
    )

    df_storage = client.query(query_storage_legacy).to_dataframe(
        create_bqstorage_client=False
    )
    storage_source = "__TABLES__"
    write_log("Consulta alternativa de volume executada via __TABLES__.")

display(df_storage)

print(f"Fonte usada para volume das tabelas: {storage_source}")

2026-07-12 21:01:20 | WARNING | Consulta via INFORMATION_SCHEMA.TABLE_STORAGE falhou. Usando alternativa __TABLES__. Erro: 404 Not found: Dataset basedosdados:br_inep_avaliacao_alfabetizacao.INFORMATION_SCHEMA was not found in location US; reason: notFound, message: Not found: Dataset basedosdados:br_inep_avaliacao_alfabe
2026-07-12 21:01:21 | INFO | Consulta alternativa de volume executada via __TABLES__.


,table_name,total_rows,tamanho_mb
0,alunos,3867999,256.10
1,municipio,23995,1.75
2,meta_alfabetizacao_municipio,10704,1.10
3,uf,145,0.01
4,meta_alfabetizacao_uf,81,0.01
5,dicionario,27,0.00
6,meta_alfabetizacao_brasil,3,0.00


Fonte usada para volume das tabelas: __TABLES__


## Validação das tabelas obrigatórias

In [12]:
# Validação das tabelas obrigatórias do desafio

expected_tables = {
    "alunos",
    "dicionario",
    "meta_alfabetizacao_brasil",
    "meta_alfabetizacao_uf",
    "meta_alfabetizacao_municipio",
    "municipio",
    "uf",
}

found_tables = set(df_tables["table_name"].tolist())

missing_tables = expected_tables - found_tables
extra_tables = found_tables - expected_tables

validation_result = {
    "expected_tables": sorted(list(expected_tables)),
    "found_tables": sorted(list(found_tables)),
    "missing_tables": sorted(list(missing_tables)),
    "extra_tables": sorted(list(extra_tables)),
    "status": "approved" if not missing_tables else "failed"
}

display(pd.DataFrame([validation_result]))

if missing_tables:
    write_log(f"Tabelas obrigatórias ausentes: {missing_tables}", level="ERROR")
else:
    write_log("Todas as tabelas obrigatórias foram encontradas.")

,expected_tables,found_tables,missing_tables,extra_tables,status
0,"[alunos, dicionario, meta_alfabetizacao_brasil...","[alunos, dicionario, meta_alfabetizacao_brasil...",[],[],approved


2026-07-12 21:01:22 | INFO | Todas as tabelas obrigatórias foram encontradas.


## Salvar metadados no ambiente do Colab

In [13]:
# Salvando metadados de descoberta em arquivos locais do Colab

metadata_path = BRONZE_BATCH_PATH / "metadata"
metadata_path.mkdir(parents=True, exist_ok=True)

df_tables.to_csv(metadata_path / "dataset_tables.csv", index=False)
df_columns.to_csv(metadata_path / "dataset_columns.csv", index=False)
df_storage.to_csv(metadata_path / "dataset_storage.csv", index=False)

with open(metadata_path / "table_validation.json", "w", encoding="utf-8") as file:
    json.dump(validation_result, file, ensure_ascii=False, indent=4)

write_log("Metadados do dataset salvos na camada Bronze/metadata.")

2026-07-12 21:01:22 | INFO | Metadados do dataset salvos na camada Bronze/metadata.


## Verificar arquivos gerados

In [14]:
# Verificação dos arquivos gerados

for root, dirs, files in os.walk(BASE_PATH):
    level = root.replace(str(BASE_PATH), "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")

    subindent = " " * 2 * (level + 1)
    for file in files:
        print(f"{subindent}{file}")

techchallenge-fase2-pipeline-alfabetizacao/
  data/
    bronze/
      batch/
        metadata/
          table_validation.json
          dataset_tables.csv
          dataset_storage.csv
          dataset_columns.csv
      streaming/
    gold/
    silver/
  logs/
    pipeline_execution.log


## Nesta etapa, realizamos a ingestão batch dos dados públicos da Base dos Dados, utilizando o BigQuery Sandbox como componente cloud.

### A camada Bronze recebe os dados em formato bruto ou minimamente controlado, preservando a origem e permitindo rastreabilidade.

### Estratégia adotada:

- ingestão integral das tabelas pequenas;
- ingestão controlada da tabela `alunos`, devido ao maior volume;
- geração de uma amostra de alunos;
- geração de uma visão agregada de alunos por ano, município, rede e série;
- registro de manifesto da ingestão;
- registro de logs de execução.

### Essa abordagem segue a restrição acadêmica de custo zero e evita consumo desnecessário de processamento.

## Configuração da ingestão Batch

In [15]:
# Configuração da ingestão Batch

BATCH_OUTPUT_PATH = BRONZE_BATCH_PATH

SMALL_TABLES = [
    "meta_alfabetizacao_brasil",
    "meta_alfabetizacao_uf",
    "meta_alfabetizacao_municipio",
    "municipio",
    "uf",
    "dicionario",
]

ALUNOS_SAMPLE_SIZE = 100000

batch_manifest = []

write_log(f"Tabelas pequenas configuradas para ingestão integral: {SMALL_TABLES}")
write_log(f"Tamanho da amostra da tabela alunos: {ALUNOS_SAMPLE_SIZE}")

2026-07-12 21:01:22 | INFO | Tabelas pequenas configuradas para ingestão integral: ['meta_alfabetizacao_brasil', 'meta_alfabetizacao_uf', 'meta_alfabetizacao_municipio', 'municipio', 'uf', 'dicionario']
2026-07-12 21:01:22 | INFO | Tamanho da amostra da tabela alunos: 100000


## Função auxiliar para executar query e salvar CSV

In [16]:
# Função auxiliar para executar consultas BigQuery e salvar resultado em CSV

def run_query_to_csv(query: str, output_file: Path, description: str):
    """
    Executa uma consulta BigQuery, converte o resultado para DataFrame
    e salva em CSV na camada Bronze.
    """

    start_time = time.perf_counter()
    started_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    write_log(f"Iniciando extração: {description}")

    try:
        job = client.query(query)
        df = job.to_dataframe(create_bqstorage_client=False)

        output_file.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(output_file, index=False, encoding="utf-8")

        elapsed_seconds = round(time.perf_counter() - start_time, 2)
        finished_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        file_size_mb = round(output_file.stat().st_size / 1024 / 1024, 4)

        manifest_item = {
            "description": description,
            "output_file": str(output_file),
            "rows": int(len(df)),
            "columns": int(len(df.columns)),
            "bytes_processed": int(job.total_bytes_processed or 0),
            "file_size_mb": file_size_mb,
            "started_at": started_at,
            "finished_at": finished_at,
            "elapsed_seconds": elapsed_seconds,
            "status": "success"
        }

        batch_manifest.append(manifest_item)

        write_log(
            f"Extração concluída: {description} | "
            f"linhas={len(df)} | colunas={len(df.columns)} | "
            f"arquivo={output_file.name} | tamanho_mb={file_size_mb}"
        )

        return df

    except Exception as error:
        elapsed_seconds = round(time.perf_counter() - start_time, 2)
        finished_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        manifest_item = {
            "description": description,
            "output_file": str(output_file),
            "rows": 0,
            "columns": 0,
            "bytes_processed": 0,
            "file_size_mb": 0,
            "started_at": started_at,
            "finished_at": finished_at,
            "elapsed_seconds": elapsed_seconds,
            "status": "failed",
            "error": str(error)
        }

        batch_manifest.append(manifest_item)

        write_log(f"Erro na extração: {description} | erro={error}", level="ERROR")
        raise

## Ingestão integral das tabelas pequenas

In [17]:
# Ingestão integral das tabelas pequenas para a camada Bronze

bronze_dataframes = {}

for table_name in SMALL_TABLES:
    table_id = TABLES[table_name]

    query = f"""
    SELECT *
    FROM `{table_id}`
    """

    output_file = BATCH_OUTPUT_PATH / f"{table_name}.csv"

    df = run_query_to_csv(
        query=query,
        output_file=output_file,
        description=f"Ingestão integral da tabela {table_name}"
    )

    bronze_dataframes[table_name] = df

write_log("Ingestão integral das tabelas pequenas concluída.")

2026-07-12 21:01:22 | INFO | Iniciando extração: Ingestão integral da tabela meta_alfabetizacao_brasil
2026-07-12 21:01:23 | INFO | Extração concluída: Ingestão integral da tabela meta_alfabetizacao_brasil | linhas=3 | colunas=11 | arquivo=meta_alfabetizacao_brasil.csv | tamanho_mb=0.0004
2026-07-12 21:01:23 | INFO | Iniciando extração: Ingestão integral da tabela meta_alfabetizacao_uf
2026-07-12 21:01:24 | INFO | Extração concluída: Ingestão integral da tabela meta_alfabetizacao_uf | linhas=81 | colunas=12 | arquivo=meta_alfabetizacao_uf.csv | tamanho_mb=0.005
2026-07-12 21:01:24 | INFO | Iniciando extração: Ingestão integral da tabela meta_alfabetizacao_municipio
2026-07-12 21:01:28 | INFO | Extração concluída: Ingestão integral da tabela meta_alfabetizacao_municipio | linhas=10704 | colunas=13 | arquivo=meta_alfabetizacao_municipio.csv | tamanho_mb=0.7712
2026-07-12 21:01:28 | INFO | Iniciando extração: Ingestão integral da tabela municipio
2026-07-12 21:01:32 | INFO | Extração conc

## Ingestão controlada da tabela alunos: amostra

In [18]:
# Ingestão controlada da tabela alunos
# A tabela alunos possui maior volume. Para custo zero, usamos amostra controlada.

query_alunos_sample = f"""
SELECT
  ano,
  id_municipio,
  id_escola,
  id_aluno,
  caderno,
  serie,
  rede,
  presenca,
  preenchimento_caderno,
  alfabetizado,
  proficiencia,
  peso_aluno
FROM
  `{TABLES["alunos"]}`
LIMIT {ALUNOS_SAMPLE_SIZE}
"""

alunos_sample_file = BATCH_OUTPUT_PATH / "alunos_sample.csv"

df_alunos_sample = run_query_to_csv(
    query=query_alunos_sample,
    output_file=alunos_sample_file,
    description=f"Ingestão controlada da tabela alunos com LIMIT {ALUNOS_SAMPLE_SIZE}"
)

display(df_alunos_sample.head())

write_log("Amostra da tabela alunos gerada com sucesso.")

2026-07-12 21:01:34 | INFO | Iniciando extração: Ingestão controlada da tabela alunos com LIMIT 100000
2026-07-12 21:01:46 | INFO | Extração concluída: Ingestão controlada da tabela alunos com LIMIT 100000 | linhas=100000 | colunas=12 | arquivo=alunos_sample.csv | tamanho_mb=5.2326


,ano,id_municipio,id_escola,id_aluno,caderno,serie,rede,presenca,preenchimento_caderno,alfabetizado,proficiencia,peso_aluno
0,2023,1300029,60000485,13000208,1,2,2,0,0,0,NaN,NaN
1,2023,1301852,60000574,13011384,1,2,3,0,0,0,NaN,NaN
2,2023,1302603,60001009,13049504,1,2,2,0,0,0,NaN,NaN
3,2023,1701903,60004032,17013597,1,2,3,0,0,0,NaN,NaN
4,2023,2109502,60005472,21061500,1,2,3,0,0,0,NaN,NaN


2026-07-12 21:01:46 | INFO | Amostra da tabela alunos gerada com sucesso.


## Ingestão agregada da tabela alunos

In [19]:
# Ingestão agregada da tabela alunos
# Esta visão reduz o volume e permite análises por município, ano, rede e série.

query_alunos_agregado = f"""
SELECT
  ano,
  id_municipio,
  rede,
  serie,
  COUNT(*) AS total_alunos,
  COUNTIF(proficiencia IS NOT NULL) AS total_com_proficiencia,
  AVG(proficiencia) AS media_proficiencia,
  MIN(proficiencia) AS menor_proficiencia,
  MAX(proficiencia) AS maior_proficiencia,
  COUNTIF(alfabetizado IS NOT NULL) AS total_com_status_alfabetizacao,
  COUNT(DISTINCT alfabetizado) AS qtd_status_alfabetizacao,
  COUNTIF(presenca IS NOT NULL) AS total_com_status_presenca,
  AVG(peso_aluno) AS media_peso_aluno
FROM
  `{TABLES["alunos"]}`
GROUP BY
  ano,
  id_municipio,
  rede,
  serie
ORDER BY
  ano,
  id_municipio,
  rede,
  serie
"""

alunos_agregado_file = BATCH_OUTPUT_PATH / "alunos_agregado.csv"

df_alunos_agregado = run_query_to_csv(
    query=query_alunos_agregado,
    output_file=alunos_agregado_file,
    description="Ingestão agregada da tabela alunos por ano, município, rede e série"
)

display(df_alunos_agregado.head())

write_log("Visão agregada da tabela alunos gerada com sucesso.")

2026-07-12 21:01:46 | INFO | Iniciando extração: Ingestão agregada da tabela alunos por ano, município, rede e série
2026-07-12 21:01:50 | INFO | Extração concluída: Ingestão agregada da tabela alunos por ano, município, rede e série | linhas=12431 | colunas=13 | arquivo=alunos_agregado.csv | tamanho_mb=1.0165


,ano,id_municipio,rede,serie,total_alunos,total_com_proficiencia,media_proficiencia,menor_proficiencia,maior_proficiencia,total_com_status_alfabetizacao,qtd_status_alfabetizacao,total_com_status_presenca,media_peso_aluno
0,2023,1100015,3,2,254,227,759.937724,640.637477,846.516484,254,2,254,1.118943
1,2023,1100023,3,2,1361,1222,757.466012,602.616267,853.508475,1361,2,1361,1.114386
2,2023,1100031,3,2,84,76,767.675528,650.033845,848.987394,84,2,84,1.105263
3,2023,1100049,2,2,227,200,762.040941,617.300913,846.516484,227,2,227,1.135000
4,2023,1100049,3,2,726,613,756.053475,623.151566,848.987394,726,2,726,1.185301


2026-07-12 21:01:50 | INFO | Visão agregada da tabela alunos gerada com sucesso.


## Gerar manifesto da ingestão Batch

In [20]:
# Geração do manifesto da ingestão Batch

manifest_file = BATCH_OUTPUT_PATH / "batch_ingestion_manifest.json"

with open(manifest_file, "w", encoding="utf-8") as file:
    json.dump(batch_manifest, file, ensure_ascii=False, indent=4)

write_log(f"Manifesto da ingestão Batch gerado: {manifest_file}")

df_manifest = pd.DataFrame(batch_manifest)
display(df_manifest)

2026-07-12 21:01:50 | INFO | Manifesto da ingestão Batch gerado: /content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/batch/batch_ingestion_manifest.json


,description,output_file,rows,columns,bytes_processed,file_size_mb,started_at,finished_at,elapsed_seconds,status
0,Ingestão integral da tabela meta_alfabetizacao...,/content/techchallenge-fase2-pipeline-alfabeti...,3,11,270,0.0004,2026-07-12 21:01:22,2026-07-12 21:01:23,1.51,success
1,Ingestão integral da tabela meta_alfabetizacao_uf,/content/techchallenge-fase2-pipeline-alfabeti...,81,12,7374,0.0050,2026-07-12 21:01:23,2026-07-12 21:01:24,1.21,success
2,Ingestão integral da tabela meta_alfabetizacao...,/content/techchallenge-fase2-pipeline-alfabeti...,10704,13,1151232,0.7712,2026-07-12 21:01:24,2026-07-12 21:01:28,3.44,success
3,Ingestão integral da tabela municipio,/content/techchallenge-fase2-pipeline-alfabeti...,23995,15,1832061,1.3642,2026-07-12 21:01:28,2026-07-12 21:01:32,4.15,success
4,Ingestão integral da tabela uf,/content/techchallenge-fase2-pipeline-alfabeti...,145,15,10330,0.0079,2026-07-12 21:01:32,2026-07-12 21:01:33,0.92,success
5,Ingestão integral da tabela dicionario,/content/techchallenge-fase2-pipeline-alfabeti...,27,5,1174,0.0010,2026-07-12 21:01:33,2026-07-12 21:01:34,1.14,success
6,Ingestão controlada da tabela alunos com LIMIT...,/content/techchallenge-fase2-pipeline-alfabeti...,100000,12,268545240,5.2326,2026-07-12 21:01:34,2026-07-12 21:01:46,12.07,success
7,"Ingestão agregada da tabela alunos por ano, mu...",/content/techchallenge-fase2-pipeline-alfabeti...,12431,13,165846547,1.0165,2026-07-12 21:01:46,2026-07-12 21:01:50,4.02,success


## Validação dos arquivos gerados na Bronze

In [21]:
# Validação dos arquivos gerados na camada Bronze Batch

expected_bronze_files = [
    "meta_alfabetizacao_brasil.csv",
    "meta_alfabetizacao_uf.csv",
    "meta_alfabetizacao_municipio.csv",
    "municipio.csv",
    "uf.csv",
    "dicionario.csv",
    "alunos_sample.csv",
    "alunos_agregado.csv",
    "batch_ingestion_manifest.json",
]

bronze_validation = []

for file_name in expected_bronze_files:
    file_path = BATCH_OUTPUT_PATH / file_name

    bronze_validation.append({
        "file_name": file_name,
        "exists": file_path.exists(),
        "size_mb": round(file_path.stat().st_size / 1024 / 1024, 4) if file_path.exists() else 0
    })

df_bronze_validation = pd.DataFrame(bronze_validation)

display(df_bronze_validation)

if df_bronze_validation["exists"].all():
    write_log("Validação da camada Bronze Batch aprovada: todos os arquivos esperados foram gerados.")
else:
    missing = df_bronze_validation[df_bronze_validation["exists"] == False]["file_name"].tolist()
    write_log(f"Arquivos ausentes na Bronze Batch: {missing}", level="ERROR")

,file_name,exists,size_mb
0,meta_alfabetizacao_brasil.csv,True,0.0004
1,meta_alfabetizacao_uf.csv,True,0.0050
2,meta_alfabetizacao_municipio.csv,True,0.7712
3,municipio.csv,True,1.3642
4,uf.csv,True,0.0079
5,dicionario.csv,True,0.0010
6,alunos_sample.csv,True,5.2326
7,alunos_agregado.csv,True,1.0165
8,batch_ingestion_manifest.json,True,0.0036


2026-07-12 21:01:50 | INFO | Validação da camada Bronze Batch aprovada: todos os arquivos esperados foram gerados.


## Resumo da ingestão Batch

In [22]:
# Resumo executivo da ingestão Batch

total_files = len(expected_bronze_files)
total_rows = int(df_manifest["rows"].sum())
total_size_mb = round(df_bronze_validation["size_mb"].sum(), 4)
total_bytes_processed = int(df_manifest["bytes_processed"].sum())

summary_batch = {
    "total_files_generated": total_files,
    "total_rows_extracted": total_rows,
    "total_size_mb": total_size_mb,
    "total_bytes_processed": total_bytes_processed,
    "status": "approved" if df_bronze_validation["exists"].all() else "failed"
}

display(pd.DataFrame([summary_batch]))

write_log(
    f"Resumo Batch: arquivos={total_files}, linhas={total_rows}, "
    f"tamanho_mb={total_size_mb}, bytes_processados={total_bytes_processed}, "
    f"status={summary_batch['status']}"
)

,total_files_generated,total_rows_extracted,total_size_mb,total_bytes_processed,status
0,9,147386,8.4024,437394228,approved


2026-07-12 21:01:50 | INFO | Resumo Batch: arquivos=9, linhas=147386, tamanho_mb=8.4024, bytes_processados=437394228, status=approved


## Listar arquivos gerados

In [23]:
# Listagem dos arquivos gerados no ambiente Colab

!find /content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/batch -maxdepth 3 -type f

/content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/batch/dicionario.csv
/content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/batch/metadata/table_validation.json
/content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/batch/metadata/dataset_tables.csv
/content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/batch/metadata/dataset_storage.csv
/content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/batch/metadata/dataset_columns.csv
/content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/batch/meta_alfabetizacao_uf.csv
/content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/batch/municipio.csv
/content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/batch/meta_alfabetizacao_municipio.csv
/content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/batch/uf.csv
/content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/batch/batch_ingestion_manifest.json
/content/techchallenge-fase2-pipeline-alfabetizacao/data/b

## Nesta etapa, os dados da camada Bronze são tratados e padronizados para compor a camada Silver.

### A camada Silver aplica:

- padronização de nomes e tipos de colunas;
- normalização de códigos de município e UF;
- remoção de duplicidades;
- tratamento estrutural de valores nulos;
- validação de campos obrigatórios;
- validação de faixas esperadas para percentuais;
- geração de relatório de qualidade;
- salvamento em formato Parquet.

### O uso de Parquet nas camadas tratadas contribui para otimização de armazenamento e leitura, reforçando a estratégia de FinOps do projeto.

### Configuração da camada Silver

In [24]:
# Configuração da camada Silver

SILVER_OUTPUT_PATH = SILVER_PATH
SILVER_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

silver_manifest = []
silver_quality_report = []

write_log("Passo 7 iniciado: transformação da camada Bronze para Silver.")

2026-07-12 21:01:50 | INFO | Passo 7 iniciado: transformação da camada Bronze para Silver.


Funções auxiliares da Silver

In [25]:
# Funções auxiliares para transformação e validação da camada Silver

def read_bronze_csv(file_name: str) -> pd.DataFrame:
    """
    Lê um arquivo CSV da camada Bronze Batch.
    Alguns campos são forçados como string para preservar chaves e códigos.
    """
    file_path = BATCH_OUTPUT_PATH / file_name

    dtype_map = {
        "id_municipio": "string",
        "id_escola": "string",
        "id_aluno": "string",
        "sigla_uf": "string",
        "rede": "string",
        "serie": "string",
        "caderno": "string",
        "presenca": "string",
        "preenchimento_caderno": "string",
        "alfabetizado": "string",
        "id_tabela": "string",
        "nome_coluna": "string",
        "chave": "string",
        "cobertura_temporal": "string",
        "valor": "string",
    }

    df = pd.read_csv(file_path, dtype=dtype_map, low_memory=False)
    write_log(f"Arquivo Bronze lido: {file_name} | linhas={len(df)} | colunas={len(df.columns)}")

    return df


def normalize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """
    Padroniza nomes de colunas para minúsculas e sem espaços.
    """
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )
    return df


def normalize_string_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove espaços extras em colunas textuais.
    """
    df = df.copy()

    for column in df.columns:
        if pd.api.types.is_string_dtype(df[column]) or df[column].dtype == "object":
            df[column] = df[column].astype("string").str.strip()

    return df


def normalize_keys(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normaliza chaves principais como id_municipio e sigla_uf.
    """
    df = df.copy()

    if "id_municipio" in df.columns:
        df["id_municipio"] = (
            df["id_municipio"]
            .astype("string")
            .str.replace(r"\.0$", "", regex=True)
            .str.zfill(7)
        )

    if "sigla_uf" in df.columns:
        df["sigla_uf"] = df["sigla_uf"].astype("string").str.upper().str.strip()

    if "rede" in df.columns:
        df["rede"] = df["rede"].astype("string").str.replace(r"\.0$", "", regex=True).str.strip()

    if "serie" in df.columns:
        df["serie"] = df["serie"].astype("string").str.replace(r"\.0$", "", regex=True).str.strip()

    return df


def convert_numeric_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Converte colunas numéricas conhecidas para os tipos adequados.
    """
    df = df.copy()

    integer_columns = [
        "ano",
        "nivel_alfabetizacao",
        "total_alunos",
        "total_com_proficiencia",
        "total_com_status_alfabetizacao",
        "qtd_status_alfabetizacao",
        "total_com_status_presenca",
    ]

    float_prefixes = [
        "taxa_",
        "meta_",
        "percentual_",
        "media_",
        "proporcao_",
        "proficiencia",
        "peso_",
        "menor_",
        "maior_",
    ]

    for column in integer_columns:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="coerce").astype("Int64")

    for column in df.columns:
        if any(column.startswith(prefix) for prefix in float_prefixes):
            df[column] = pd.to_numeric(df[column], errors="coerce")

    return df


def add_processing_metadata(df: pd.DataFrame, source_table: str) -> pd.DataFrame:
    """
    Adiciona metadados técnicos da transformação.
    """
    df = df.copy()
    df["source_table"] = source_table
    df["processed_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    df["layer"] = "silver"
    return df


def remove_duplicates(df: pd.DataFrame, subset_keys: list | None = None) -> tuple[pd.DataFrame, int]:
    """
    Remove duplicidades. Quando subset_keys é informado, usa as chaves de negócio.
    """
    df = df.copy()
    rows_before = len(df)

    if subset_keys:
        available_keys = [col for col in subset_keys if col in df.columns]
        df = df.drop_duplicates(subset=available_keys)
    else:
        df = df.drop_duplicates()

    rows_after = len(df)
    duplicates_removed = rows_before - rows_after

    return df, duplicates_removed


def validate_required_fields(df: pd.DataFrame, table_name: str, required_fields: list) -> dict:
    """
    Valida nulos em campos obrigatórios.
    """
    result = {
        "table_name": table_name,
        "validation_type": "required_fields",
        "status": "approved",
        "errors": {}
    }

    for field in required_fields:
        if field in df.columns:
            null_count = int(df[field].isna().sum())
            if null_count > 0:
                result["errors"][field] = null_count

    if result["errors"]:
        result["status"] = "warning"

    return result


def validate_percentage_ranges(df: pd.DataFrame, table_name: str) -> dict:
    """
    Valida campos percentuais esperados entre 0 e 100.
    Não altera os dados; apenas registra possíveis inconsistências.
    """
    percentage_columns = [
        col for col in df.columns
        if col.startswith("taxa_")
        or col.startswith("meta_alfabetizacao_")
        or col.startswith("percentual_")
        or col.startswith("proporcao_")
    ]

    result = {
        "table_name": table_name,
        "validation_type": "percentage_range_0_100",
        "status": "approved",
        "errors": {}
    }

    for column in percentage_columns:
        invalid_count = int(((df[column] < 0) | (df[column] > 100)).sum(skipna=True))
        if invalid_count > 0:
            result["errors"][column] = invalid_count

    if result["errors"]:
        result["status"] = "warning"

    return result


def transform_to_silver(
    source_file: str,
    output_file: str,
    source_table: str,
    required_fields: list,
    duplicate_keys: list | None = None
) -> pd.DataFrame:
    """
    Executa o fluxo completo de transformação Bronze -> Silver.
    """
    start_time = time.perf_counter()
    started_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    write_log(f"Iniciando transformação Silver: {source_file}")

    df = read_bronze_csv(source_file)

    rows_before = len(df)
    columns_before = len(df.columns)

    df = normalize_column_names(df)
    df = normalize_string_columns(df)
    df = normalize_keys(df)
    df = convert_numeric_columns(df)

    df, duplicates_removed = remove_duplicates(df, duplicate_keys)

    required_validation = validate_required_fields(df, output_file, required_fields)
    percentage_validation = validate_percentage_ranges(df, output_file)

    silver_quality_report.append(required_validation)
    silver_quality_report.append(percentage_validation)

    df = add_processing_metadata(df, source_table)

    output_path = SILVER_OUTPUT_PATH / output_file
    df.to_parquet(output_path, index=False)

    elapsed_seconds = round(time.perf_counter() - start_time, 2)
    finished_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    file_size_mb = round(output_path.stat().st_size / 1024 / 1024, 4)

    manifest_item = {
        "source_file": source_file,
        "output_file": str(output_path),
        "source_table": source_table,
        "rows_before": int(rows_before),
        "rows_after": int(len(df)),
        "columns_before": int(columns_before),
        "columns_after": int(len(df.columns)),
        "duplicates_removed": int(duplicates_removed),
        "file_size_mb": file_size_mb,
        "started_at": started_at,
        "finished_at": finished_at,
        "elapsed_seconds": elapsed_seconds,
        "status": "success"
    }

    silver_manifest.append(manifest_item)

    write_log(
        f"Transformação Silver concluída: {output_file} | "
        f"linhas={len(df)} | duplicidades_removidas={duplicates_removed} | "
        f"tamanho_mb={file_size_mb}"
    )

    return df

### Transformar tabelas de metas

In [26]:
# Transformação das tabelas de metas para Silver

df_silver_meta_brasil = transform_to_silver(
    source_file="meta_alfabetizacao_brasil.csv",
    output_file="silver_meta_alfabetizacao_brasil.parquet",
    source_table="meta_alfabetizacao_brasil",
    required_fields=["ano", "rede", "taxa_alfabetizacao"],
    duplicate_keys=["ano", "rede"]
)

df_silver_meta_uf = transform_to_silver(
    source_file="meta_alfabetizacao_uf.csv",
    output_file="silver_meta_alfabetizacao_uf.parquet",
    source_table="meta_alfabetizacao_uf",
    required_fields=["ano", "sigla_uf", "rede", "taxa_alfabetizacao"],
    duplicate_keys=["ano", "sigla_uf", "rede"]
)

df_silver_meta_municipio = transform_to_silver(
    source_file="meta_alfabetizacao_municipio.csv",
    output_file="silver_meta_alfabetizacao_municipio.parquet",
    source_table="meta_alfabetizacao_municipio",
    required_fields=["ano", "id_municipio", "rede", "taxa_alfabetizacao"],
    duplicate_keys=["ano", "id_municipio", "rede"]
)

display(df_silver_meta_brasil.head())
display(df_silver_meta_uf.head())
display(df_silver_meta_municipio.head())

2026-07-12 21:01:50 | INFO | Iniciando transformação Silver: meta_alfabetizacao_brasil.csv
2026-07-12 21:01:50 | INFO | Arquivo Bronze lido: meta_alfabetizacao_brasil.csv | linhas=3 | colunas=11
2026-07-12 21:01:51 | INFO | Transformação Silver concluída: silver_meta_alfabetizacao_brasil.parquet | linhas=3 | duplicidades_removidas=0 | tamanho_mb=0.009
2026-07-12 21:01:51 | INFO | Iniciando transformação Silver: meta_alfabetizacao_uf.csv
2026-07-12 21:01:51 | INFO | Arquivo Bronze lido: meta_alfabetizacao_uf.csv | linhas=81 | colunas=12
2026-07-12 21:01:51 | INFO | Transformação Silver concluída: silver_meta_alfabetizacao_uf.parquet | linhas=81 | duplicidades_removidas=0 | tamanho_mb=0.0118
2026-07-12 21:01:51 | INFO | Iniciando transformação Silver: meta_alfabetizacao_municipio.csv
2026-07-12 21:01:51 | INFO | Arquivo Bronze lido: meta_alfabetizacao_municipio.csv | linhas=10704 | colunas=13
2026-07-12 21:01:51 | INFO | Transformação Silver concluída: silver_meta_alfabetizacao_municipio

,ano,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,percentual_participacao,source_table,processed_at,layer
0,2025,Pública,66.0,60.0,64.00,67.00,71.00,74.00,77.00,80.0,88.00,meta_alfabetizacao_brasil,2026-07-12 21:01:50,silver
1,2024,Pública,59.2,59.9,63.77,67.47,70.97,74.23,77.24,80.0,87.37,meta_alfabetizacao_brasil,2026-07-12 21:01:50,silver
2,2023,Pública,55.9,59.9,63.77,67.47,70.97,74.23,77.24,80.0,86.00,meta_alfabetizacao_brasil,2026-07-12 21:01:50,silver


,ano,sigla_uf,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,percentual_participacao,source_table,processed_at,layer
0,2024,RR,Pública,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,meta_alfabetizacao_uf,2026-07-12 21:01:51,silver
1,2023,RR,Pública,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,meta_alfabetizacao_uf,2026-07-12 21:01:51,silver
2,2024,SE,Pública,38.39,38.3,45.9,53.6,61.2,68.3,74.6,80.0,92.84,meta_alfabetizacao_uf,2026-07-12 21:01:51,silver
3,2023,SE,Pública,31.30,38.3,45.9,53.6,61.2,68.3,74.6,80.0,88.34,meta_alfabetizacao_uf,2026-07-12 21:01:51,silver
4,2025,SE,Pública,50.00,38.0,46.0,54.0,61.0,68.0,75.0,80.0,87.00,meta_alfabetizacao_uf,2026-07-12 21:01:51,silver


,ano,id_municipio,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,nivel_alfabetizacao,percentual_participacao,source_table,processed_at,layer
0,2023,4301750,Municipal,NaN,NaN,14.05,23.65,37.00,52.68,67.85,80.0,<NA>,NaN,meta_alfabetizacao_municipio,2026-07-12 21:01:51,silver
1,2024,4301750,Municipal,4.40,NaN,14.05,23.65,37.00,52.68,67.85,80.0,0,92.59,meta_alfabetizacao_municipio,2026-07-12 21:01:51,silver
2,2024,2406908,Municipal,42.86,7.94,14.05,23.65,37.00,52.68,67.85,80.0,1,84.00,meta_alfabetizacao_municipio,2026-07-12 21:01:51,silver
3,2023,2406908,Municipal,4.40,7.94,14.05,23.65,37.00,52.68,67.85,80.0,0,82.14,meta_alfabetizacao_municipio,2026-07-12 21:01:51,silver
4,2023,1718501,Municipal,4.60,8.25,14.48,24.16,37.49,53.03,68.00,80.0,0,95.65,meta_alfabetizacao_municipio,2026-07-12 21:01:51,silver


### Transformar indicadores por município e UF

In [27]:
# Transformação das tabelas de indicadores por município e UF para Silver

df_silver_indicador_municipio = transform_to_silver(
    source_file="municipio.csv",
    output_file="silver_indicador_municipio.parquet",
    source_table="municipio",
    required_fields=["ano", "id_municipio", "serie", "rede", "taxa_alfabetizacao"],
    duplicate_keys=["ano", "id_municipio", "serie", "rede"]
)

df_silver_indicador_uf = transform_to_silver(
    source_file="uf.csv",
    output_file="silver_indicador_uf.parquet",
    source_table="uf",
    required_fields=["ano", "sigla_uf", "serie", "rede", "taxa_alfabetizacao"],
    duplicate_keys=["ano", "sigla_uf", "serie", "rede"]
)

display(df_silver_indicador_municipio.head())
display(df_silver_indicador_uf.head())

2026-07-12 21:01:51 | INFO | Iniciando transformação Silver: municipio.csv
2026-07-12 21:01:51 | INFO | Arquivo Bronze lido: municipio.csv | linhas=23995 | colunas=15
2026-07-12 21:01:51 | INFO | Transformação Silver concluída: silver_indicador_municipio.parquet | linhas=23995 | duplicidades_removidas=0 | tamanho_mb=0.4763
2026-07-12 21:01:51 | INFO | Iniciando transformação Silver: uf.csv
2026-07-12 21:01:51 | INFO | Arquivo Bronze lido: uf.csv | linhas=145 | colunas=15
2026-07-12 21:01:51 | INFO | Transformação Silver concluída: silver_indicador_uf.parquet | linhas=145 | duplicidades_removidas=0 | tamanho_mb=0.017


,ano,id_municipio,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8,source_table,processed_at,layer
0,2023,1100031,2,3,69.10,767.8763,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,municipio,2026-07-12 21:01:51,silver
1,2023,1100072,2,3,58.20,747.8918,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,municipio,2026-07-12 21:01:51,silver
2,2023,1100189,2,5,69.73,762.4062,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,municipio,2026-07-12 21:01:51,silver
3,2023,1101609,2,3,50.70,745.6802,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,municipio,2026-07-12 21:01:51,silver
4,2023,1101807,2,3,55.69,752.3724,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,municipio,2026-07-12 21:01:51,silver


,ano,sigla_uf,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8,source_table,processed_at,layer
0,2023,AM,2,3,49.20,733.6637,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,uf,2026-07-12 21:01:51,silver
1,2023,PB,2,2,55.23,744.8152,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,uf,2026-07-12 21:01:51,silver
2,2023,PR,2,5,73.12,757.2146,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,uf,2026-07-12 21:01:51,silver
3,2023,AP,2,3,41.87,732.7858,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,uf,2026-07-12 21:01:51,silver
4,2023,PE,2,5,58.95,747.4522,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,uf,2026-07-12 21:01:51,silver


### Transformar dados de alunos

In [28]:
# Transformação das tabelas derivadas de alunos para Silver

df_silver_alunos_sample = transform_to_silver(
    source_file="alunos_sample.csv",
    output_file="silver_alunos_sample.parquet",
    source_table="alunos",
    required_fields=["ano", "id_municipio", "id_escola", "id_aluno", "serie", "rede"],
    duplicate_keys=["ano", "id_aluno"]
)

df_silver_alunos_agregado = transform_to_silver(
    source_file="alunos_agregado.csv",
    output_file="silver_alunos_agregado.parquet",
    source_table="alunos_agregado",
    required_fields=["ano", "id_municipio", "rede", "serie", "total_alunos"],
    duplicate_keys=["ano", "id_municipio", "rede", "serie"]
)

display(df_silver_alunos_sample.head())
display(df_silver_alunos_agregado.head())

2026-07-12 21:01:51 | INFO | Iniciando transformação Silver: alunos_sample.csv
2026-07-12 21:01:51 | INFO | Arquivo Bronze lido: alunos_sample.csv | linhas=100000 | colunas=12
2026-07-12 21:01:53 | INFO | Transformação Silver concluída: silver_alunos_sample.parquet | linhas=100000 | duplicidades_removidas=0 | tamanho_mb=1.8235
2026-07-12 21:01:53 | INFO | Iniciando transformação Silver: alunos_agregado.csv
2026-07-12 21:01:53 | INFO | Arquivo Bronze lido: alunos_agregado.csv | linhas=12431 | colunas=13
2026-07-12 21:01:53 | INFO | Transformação Silver concluída: silver_alunos_agregado.parquet | linhas=12431 | duplicidades_removidas=0 | tamanho_mb=0.5253


,ano,id_municipio,id_escola,id_aluno,caderno,serie,rede,presenca,preenchimento_caderno,alfabetizado,proficiencia,peso_aluno,source_table,processed_at,layer
0,2023,1300029,60000485,13000208,1,2,2,0,0,0,NaN,NaN,alunos,2026-07-12 21:01:53,silver
1,2023,1301852,60000574,13011384,1,2,3,0,0,0,NaN,NaN,alunos,2026-07-12 21:01:53,silver
2,2023,1302603,60001009,13049504,1,2,2,0,0,0,NaN,NaN,alunos,2026-07-12 21:01:53,silver
3,2023,1701903,60004032,17013597,1,2,3,0,0,0,NaN,NaN,alunos,2026-07-12 21:01:53,silver
4,2023,2109502,60005472,21061500,1,2,3,0,0,0,NaN,NaN,alunos,2026-07-12 21:01:53,silver


,ano,id_municipio,rede,serie,total_alunos,total_com_proficiencia,media_proficiencia,menor_proficiencia,maior_proficiencia,total_com_status_alfabetizacao,qtd_status_alfabetizacao,total_com_status_presenca,media_peso_aluno,source_table,processed_at,layer
0,2023,1100015,3,2,254,227,759.937724,640.637477,846.516484,254,2,254,1.118943,alunos_agregado,2026-07-12 21:01:53,silver
1,2023,1100023,3,2,1361,1222,757.466012,602.616267,853.508475,1361,2,1361,1.114386,alunos_agregado,2026-07-12 21:01:53,silver
2,2023,1100031,3,2,84,76,767.675528,650.033845,848.987394,84,2,84,1.105263,alunos_agregado,2026-07-12 21:01:53,silver
3,2023,1100049,2,2,227,200,762.040941,617.300913,846.516484,227,2,227,1.135000,alunos_agregado,2026-07-12 21:01:53,silver
4,2023,1100049,3,2,726,613,756.053475,623.151566,848.987394,726,2,726,1.185301,alunos_agregado,2026-07-12 21:01:53,silver


### Transformar dicionário de dados

In [29]:
# Transformação do dicionário de dados para Silver

df_silver_dicionario = transform_to_silver(
    source_file="dicionario.csv",
    output_file="silver_dicionario.parquet",
    source_table="dicionario",
    required_fields=["id_tabela", "nome_coluna", "chave", "valor"],
    duplicate_keys=["id_tabela", "nome_coluna", "chave"]
)

display(df_silver_dicionario.head(20))

2026-07-12 21:01:53 | INFO | Iniciando transformação Silver: dicionario.csv
2026-07-12 21:01:53 | INFO | Arquivo Bronze lido: dicionario.csv | linhas=27 | colunas=5
2026-07-12 21:01:53 | INFO | Transformação Silver concluída: silver_dicionario.parquet | linhas=27 | duplicidades_removidas=0 | tamanho_mb=0.005


,id_tabela,nome_coluna,chave,cobertura_temporal,valor,source_table,processed_at,layer
0,alunos,alfabetizado,0,(1),Não,dicionario,2026-07-12 21:01:53,silver
1,alunos,alfabetizado,1,(1),Sim,dicionario,2026-07-12 21:01:53,silver
2,alunos,preenchimento_caderno,0,(1),Prova não preenchida,dicionario,2026-07-12 21:01:53,silver
3,alunos,preenchimento_caderno,1,(1),Prova preenchida,dicionario,2026-07-12 21:01:53,silver
4,alunos,presenca,0,(1),Ausente,dicionario,2026-07-12 21:01:53,silver
5,alunos,presenca,1,(1),Presente,dicionario,2026-07-12 21:01:53,silver
6,uf,rede,0,(1),"Total (Federal, Estadual, Municipal e Privada)",dicionario,2026-07-12 21:01:53,silver
7,municipio,rede,0,(1),"Total (Federal, Estadual, Municipal e Privada)",dicionario,2026-07-12 21:01:53,silver
8,uf,rede,1,(1),Federal,dicionario,2026-07-12 21:01:53,silver
9,municipio,rede,1,(1),Federal,dicionario,2026-07-12 21:01:53,silver


### Salvar manifesto e relatório de qualidade da Silver

In [30]:
# Salvando manifesto e relatório de qualidade da camada Silver

silver_manifest_file = SILVER_OUTPUT_PATH / "silver_transform_manifest.json"
silver_quality_file = SILVER_OUTPUT_PATH / "silver_quality_report.json"

with open(silver_manifest_file, "w", encoding="utf-8") as file:
    json.dump(silver_manifest, file, ensure_ascii=False, indent=4)

with open(silver_quality_file, "w", encoding="utf-8") as file:
    json.dump(silver_quality_report, file, ensure_ascii=False, indent=4)

df_silver_manifest = pd.DataFrame(silver_manifest)
df_silver_quality = pd.DataFrame(silver_quality_report)

display(df_silver_manifest)
display(df_silver_quality)

write_log(f"Manifesto Silver gerado: {silver_manifest_file}")
write_log(f"Relatório de qualidade Silver gerado: {silver_quality_file}")

,source_file,output_file,source_table,rows_before,rows_after,columns_before,columns_after,duplicates_removed,file_size_mb,started_at,finished_at,elapsed_seconds,status
0,meta_alfabetizacao_brasil.csv,/content/techchallenge-fase2-pipeline-alfabeti...,meta_alfabetizacao_brasil,3,3,11,14,0,0.0090,2026-07-12 21:01:50,2026-07-12 21:01:51,0.09,success
1,meta_alfabetizacao_uf.csv,/content/techchallenge-fase2-pipeline-alfabeti...,meta_alfabetizacao_uf,81,81,12,15,0,0.0118,2026-07-12 21:01:51,2026-07-12 21:01:51,0.02,success
2,meta_alfabetizacao_municipio.csv,/content/techchallenge-fase2-pipeline-alfabeti...,meta_alfabetizacao_municipio,10704,10704,13,16,0,0.2131,2026-07-12 21:01:51,2026-07-12 21:01:51,0.08,success
3,municipio.csv,/content/techchallenge-fase2-pipeline-alfabeti...,municipio,23995,23995,15,18,0,0.4763,2026-07-12 21:01:51,2026-07-12 21:01:51,0.27,success
4,uf.csv,/content/techchallenge-fase2-pipeline-alfabeti...,uf,145,145,15,18,0,0.0170,2026-07-12 21:01:51,2026-07-12 21:01:51,0.03,success
5,alunos_sample.csv,/content/techchallenge-fase2-pipeline-alfabeti...,alunos,100000,100000,12,15,0,1.8235,2026-07-12 21:01:51,2026-07-12 21:01:53,1.78,success
6,alunos_agregado.csv,/content/techchallenge-fase2-pipeline-alfabeti...,alunos_agregado,12431,12431,13,16,0,0.5253,2026-07-12 21:01:53,2026-07-12 21:01:53,0.17,success
7,dicionario.csv,/content/techchallenge-fase2-pipeline-alfabeti...,dicionario,27,27,5,8,0,0.0050,2026-07-12 21:01:53,2026-07-12 21:01:53,0.02,success


,table_name,validation_type,status,errors
0,silver_meta_alfabetizacao_brasil.parquet,required_fields,approved,{}
1,silver_meta_alfabetizacao_brasil.parquet,percentage_range_0_100,approved,{}
2,silver_meta_alfabetizacao_uf.parquet,required_fields,warning,{'taxa_alfabetizacao': 4}
3,silver_meta_alfabetizacao_uf.parquet,percentage_range_0_100,approved,{}
4,silver_meta_alfabetizacao_municipio.parquet,required_fields,warning,{'taxa_alfabetizacao': 120}
5,silver_meta_alfabetizacao_municipio.parquet,percentage_range_0_100,approved,{}
6,silver_indicador_municipio.parquet,required_fields,approved,{}
7,silver_indicador_municipio.parquet,percentage_range_0_100,approved,{}
8,silver_indicador_uf.parquet,required_fields,approved,{}
9,silver_indicador_uf.parquet,percentage_range_0_100,approved,{}


2026-07-12 21:01:53 | INFO | Manifesto Silver gerado: /content/techchallenge-fase2-pipeline-alfabetizacao/data/silver/silver_transform_manifest.json
2026-07-12 21:01:53 | INFO | Relatório de qualidade Silver gerado: /content/techchallenge-fase2-pipeline-alfabetizacao/data/silver/silver_quality_report.json


### Validação dos arquivos Silver esperados

In [31]:
# Validação dos arquivos esperados na camada Silver

expected_silver_files = [
    "silver_meta_alfabetizacao_brasil.parquet",
    "silver_meta_alfabetizacao_uf.parquet",
    "silver_meta_alfabetizacao_municipio.parquet",
    "silver_indicador_municipio.parquet",
    "silver_indicador_uf.parquet",
    "silver_alunos_sample.parquet",
    "silver_alunos_agregado.parquet",
    "silver_dicionario.parquet",
    "silver_transform_manifest.json",
    "silver_quality_report.json",
]

silver_validation = []

for file_name in expected_silver_files:
    file_path = SILVER_OUTPUT_PATH / file_name

    silver_validation.append({
        "file_name": file_name,
        "exists": file_path.exists(),
        "size_mb": round(file_path.stat().st_size / 1024 / 1024, 4) if file_path.exists() else 0
    })

df_silver_validation = pd.DataFrame(silver_validation)

display(df_silver_validation)

if df_silver_validation["exists"].all():
    write_log("Validação da camada Silver aprovada: todos os arquivos esperados foram gerados.")
else:
    missing = df_silver_validation[df_silver_validation["exists"] == False]["file_name"].tolist()
    write_log(f"Arquivos ausentes na camada Silver: {missing}", level="ERROR")

,file_name,exists,size_mb
0,silver_meta_alfabetizacao_brasil.parquet,True,0.0090
1,silver_meta_alfabetizacao_uf.parquet,True,0.0118
2,silver_meta_alfabetizacao_municipio.parquet,True,0.2131
3,silver_indicador_municipio.parquet,True,0.4763
4,silver_indicador_uf.parquet,True,0.0170
5,silver_alunos_sample.parquet,True,1.8235
6,silver_alunos_agregado.parquet,True,0.5253
7,silver_dicionario.parquet,True,0.0050
8,silver_transform_manifest.json,True,0.0043
9,silver_quality_report.json,True,0.0027


2026-07-12 21:01:53 | INFO | Validação da camada Silver aprovada: todos os arquivos esperados foram gerados.


### Resumo da camada Silver

In [32]:
# Resumo executivo da camada Silver

total_silver_files = len(expected_silver_files)
total_silver_rows = int(df_silver_manifest["rows_after"].sum())
total_silver_size_mb = round(df_silver_validation["size_mb"].sum(), 4)

silver_summary = {
    "total_files_generated": total_silver_files,
    "total_rows_processed": total_silver_rows,
    "total_size_mb": total_silver_size_mb,
    "total_duplicates_removed": int(df_silver_manifest["duplicates_removed"].sum()),
    "status": "approved" if df_silver_validation["exists"].all() else "failed"
}

display(pd.DataFrame([silver_summary]))

write_log(
    f"Resumo Silver: arquivos={total_silver_files}, linhas_processadas={total_silver_rows}, "
    f"tamanho_mb={total_silver_size_mb}, duplicidades_removidas={silver_summary['total_duplicates_removed']}, "
    f"status={silver_summary['status']}"
)

,total_files_generated,total_rows_processed,total_size_mb,total_duplicates_removed,status
0,10,147386,3.088,0,approved


2026-07-12 21:01:53 | INFO | Resumo Silver: arquivos=10, linhas_processadas=147386, tamanho_mb=3.088, duplicidades_removidas=0, status=approved


### Listar arquivos gerados na Silver

In [33]:
# Listagem dos arquivos gerados na camada Silver

!find /content/techchallenge-fase2-pipeline-alfabetizacao/data/silver -maxdepth 2 -type f

/content/techchallenge-fase2-pipeline-alfabetizacao/data/silver/silver_meta_alfabetizacao_brasil.parquet
/content/techchallenge-fase2-pipeline-alfabetizacao/data/silver/silver_meta_alfabetizacao_municipio.parquet
/content/techchallenge-fase2-pipeline-alfabetizacao/data/silver/silver_meta_alfabetizacao_uf.parquet
/content/techchallenge-fase2-pipeline-alfabetizacao/data/silver/silver_indicador_uf.parquet
/content/techchallenge-fase2-pipeline-alfabetizacao/data/silver/silver_quality_report.json
/content/techchallenge-fase2-pipeline-alfabetizacao/data/silver/silver_transform_manifest.json
/content/techchallenge-fase2-pipeline-alfabetizacao/data/silver/silver_alunos_sample.parquet
/content/techchallenge-fase2-pipeline-alfabetizacao/data/silver/silver_alunos_agregado.parquet
/content/techchallenge-fase2-pipeline-alfabetizacao/data/silver/silver_indicador_municipio.parquet
/content/techchallenge-fase2-pipeline-alfabetizacao/data/silver/silver_dicionario.parquet


## Nesta etapa, os dados tratados da camada Silver são integrados e transformados em bases analíticas finais.

### A camada Gold tem como objetivo disponibilizar datasets prontos para:

- análise do indicador de alfabetização por município;
- comparação entre metas e resultados;
- visão consolidada por UF;
- ranking de municípios prioritários;
- evolução temporal da alfabetização;
- preparação de uma base para uso futuro em IA.

### Os arquivos Gold serão salvos em formato Parquet para manter a estratégia de otimização de armazenamento e leitura.

### Configuração da Gold

In [34]:
# Configuração da camada Gold

GOLD_OUTPUT_PATH = GOLD_PATH
GOLD_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

gold_manifest = []
gold_quality_report = []

write_log("Passo 8 iniciado: criação da camada Gold.")

2026-07-12 21:01:54 | INFO | Passo 8 iniciado: criação da camada Gold.


### Funções auxiliares da Gold

In [35]:
# Funções auxiliares para camada Gold

UF_CODE_TO_SIGLA = {
    "11": "RO", "12": "AC", "13": "AM", "14": "RR", "15": "PA", "16": "AP", "17": "TO",
    "21": "MA", "22": "PI", "23": "CE", "24": "RN", "25": "PB", "26": "PE", "27": "AL", "28": "SE", "29": "BA",
    "31": "MG", "32": "ES", "33": "RJ", "35": "SP",
    "41": "PR", "42": "SC", "43": "RS",
    "50": "MS", "51": "MT", "52": "GO", "53": "DF",
}


def read_silver_parquet(file_name: str) -> pd.DataFrame:
    """
    Lê um arquivo Parquet da camada Silver.
    """
    file_path = SILVER_OUTPUT_PATH / file_name
    df = pd.read_parquet(file_path)
    write_log(f"Arquivo Silver lido: {file_name} | linhas={len(df)} | colunas={len(df.columns)}")
    return df


def derive_sigla_uf_from_municipio(df: pd.DataFrame) -> pd.DataFrame:
    """
    Deriva sigla_uf a partir dos dois primeiros dígitos do código IBGE do município.
    """
    df = df.copy()

    if "id_municipio" in df.columns:
        df["id_municipio"] = df["id_municipio"].astype("string").str.zfill(7)
        df["id_uf"] = df["id_municipio"].str[:2]
        df["sigla_uf"] = df["id_uf"].map(UF_CODE_TO_SIGLA).astype("string")

    return df


def add_meta_reference(df: pd.DataFrame) -> pd.DataFrame:
    """
    Define a meta de referência com base no ano da linha.
    Para ano 2023, usa meta_alfabetizacao_2024, pois não há coluna meta_2023.
    Para anos acima de 2030, usa meta_alfabetizacao_2030.
    """
    df = df.copy()

    if "ano" not in df.columns:
        return df

    df["ano"] = pd.to_numeric(df["ano"], errors="coerce").astype("Int64")

    def get_reference_year(ano):
        if pd.isna(ano):
            return pd.NA
        ano = int(ano)
        if ano < 2024:
            return 2024
        if ano > 2030:
            return 2030
        return ano

    df["ano_meta_referencia"] = df["ano"].apply(get_reference_year).astype("Int64")
    df["meta_referencia"] = np.nan

    for year in range(2024, 2031):
        column = f"meta_alfabetizacao_{year}"

        if column in df.columns:
            mask = df["ano_meta_referencia"] == year
            df.loc[mask, "meta_referencia"] = df.loc[mask, column]

    if "taxa_alfabetizacao" in df.columns:
        df["distancia_meta_pontos"] = df["taxa_alfabetizacao"] - df["meta_referencia"]

        df["status_meta"] = np.select(
            [
                df["meta_referencia"].isna(),
                df["distancia_meta_pontos"] >= 0,
                df["distancia_meta_pontos"] < 0
            ],
            [
                "sem_meta_referencia",
                "atingiu_ou_superou_meta",
                "abaixo_da_meta"
            ],
            default="indefinido"
        )

    if "meta_alfabetizacao_2030" in df.columns and "taxa_alfabetizacao" in df.columns:
        df["distancia_meta_2030_pontos"] = df["taxa_alfabetizacao"] - df["meta_alfabetizacao_2030"]

    return df


def add_gold_metadata(df: pd.DataFrame, gold_dataset_name: str) -> pd.DataFrame:
    """
    Adiciona metadados técnicos da camada Gold.
    """
    df = df.copy()
    df["gold_dataset"] = gold_dataset_name
    df["processed_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    df["layer"] = "gold"
    return df


def save_gold_dataset(df: pd.DataFrame, file_name: str, description: str):
    """
    Salva dataset Gold em Parquet e registra manifesto.
    """
    start_time = time.perf_counter()
    output_path = GOLD_OUTPUT_PATH / file_name

    df.to_parquet(output_path, index=False)

    elapsed_seconds = round(time.perf_counter() - start_time, 2)
    file_size_mb = round(output_path.stat().st_size / 1024 / 1024, 4)

    manifest_item = {
        "description": description,
        "output_file": str(output_path),
        "rows": int(len(df)),
        "columns": int(len(df.columns)),
        "file_size_mb": file_size_mb,
        "elapsed_seconds": elapsed_seconds,
        "status": "success"
    }

    gold_manifest.append(manifest_item)

    write_log(
        f"Dataset Gold gerado: {file_name} | "
        f"linhas={len(df)} | colunas={len(df.columns)} | tamanho_mb={file_size_mb}"
    )

    return output_path


def validate_gold_dataset(df: pd.DataFrame, dataset_name: str, required_fields: list):
    """
    Valida campos obrigatórios e registra relatório de qualidade da Gold.
    """
    errors = {}

    for field in required_fields:
        if field in df.columns:
            null_count = int(df[field].isna().sum())
            if null_count > 0:
                errors[field] = null_count
        else:
            errors[field] = "campo_ausente"

    result = {
        "dataset_name": dataset_name,
        "validation_type": "required_fields",
        "status": "approved" if not errors else "warning",
        "errors": errors
    }

    gold_quality_report.append(result)

    return result

### Ler arquivos Silver

In [36]:
# Leitura dos arquivos da camada Silver

df_meta_brasil = read_silver_parquet("silver_meta_alfabetizacao_brasil.parquet")
df_meta_uf = read_silver_parquet("silver_meta_alfabetizacao_uf.parquet")
df_meta_municipio = read_silver_parquet("silver_meta_alfabetizacao_municipio.parquet")

df_indicador_municipio = read_silver_parquet("silver_indicador_municipio.parquet")
df_indicador_uf = read_silver_parquet("silver_indicador_uf.parquet")

df_alunos_sample = read_silver_parquet("silver_alunos_sample.parquet")
df_alunos_agregado = read_silver_parquet("silver_alunos_agregado.parquet")
df_dicionario = read_silver_parquet("silver_dicionario.parquet")

write_log("Todos os arquivos Silver necessários para a Gold foram carregados.")

2026-07-12 21:01:54 | INFO | Arquivo Silver lido: silver_meta_alfabetizacao_brasil.parquet | linhas=3 | colunas=14
2026-07-12 21:01:54 | INFO | Arquivo Silver lido: silver_meta_alfabetizacao_uf.parquet | linhas=81 | colunas=15
2026-07-12 21:01:54 | INFO | Arquivo Silver lido: silver_meta_alfabetizacao_municipio.parquet | linhas=10704 | colunas=16
2026-07-12 21:01:54 | INFO | Arquivo Silver lido: silver_indicador_municipio.parquet | linhas=23995 | colunas=18
2026-07-12 21:01:54 | INFO | Arquivo Silver lido: silver_indicador_uf.parquet | linhas=145 | colunas=18
2026-07-12 21:01:54 | INFO | Arquivo Silver lido: silver_alunos_sample.parquet | linhas=100000 | colunas=15
2026-07-12 21:01:54 | INFO | Arquivo Silver lido: silver_alunos_agregado.parquet | linhas=12431 | colunas=16
2026-07-12 21:01:54 | INFO | Arquivo Silver lido: silver_dicionario.parquet | linhas=27 | colunas=8
2026-07-12 21:01:54 | INFO | Todos os arquivos Silver necessários para a Gold foram carregados.


### Gold: indicador por município

In [37]:
# Gold 1: Indicador de alfabetização por município

df_indicador_municipio_gold = df_indicador_municipio.copy()
df_indicador_municipio_gold = derive_sigla_uf_from_municipio(df_indicador_municipio_gold)

df_meta_municipio_join = df_meta_municipio.copy()

# Evita conflito de nomes com taxa_alfabetizacao da tabela de indicador
rename_meta_cols = {
    "taxa_alfabetizacao": "taxa_alfabetizacao_meta_base",
    "percentual_participacao": "percentual_participacao_meta"
}

df_meta_municipio_join = df_meta_municipio_join.rename(
    columns={col: new_col for col, new_col in rename_meta_cols.items() if col in df_meta_municipio_join.columns}
)

join_keys_municipio = ["ano", "id_municipio", "rede"]

df_gold_indicador_municipio = df_indicador_municipio_gold.merge(
    df_meta_municipio_join,
    on=join_keys_municipio,
    how="left",
    suffixes=("", "_meta")
)

df_gold_indicador_municipio = df_gold_indicador_municipio.merge(
    df_alunos_agregado,
    on=["ano", "id_municipio", "rede", "serie"],
    how="left",
    suffixes=("", "_alunos")
)

df_gold_indicador_municipio = add_meta_reference(df_gold_indicador_municipio)
df_gold_indicador_municipio = add_gold_metadata(
    df_gold_indicador_municipio,
    "gold_indicador_municipio"
)

validate_gold_dataset(
    df_gold_indicador_municipio,
    "gold_indicador_municipio",
    ["ano", "id_municipio", "sigla_uf", "rede", "serie", "taxa_alfabetizacao"]
)

save_gold_dataset(
    df_gold_indicador_municipio,
    "gold_indicador_municipio.parquet",
    "Indicador de alfabetização por município integrado com metas e dados agregados de alunos"
)

display(df_gold_indicador_municipio.head())

2026-07-12 21:01:54 | INFO | Dataset Gold gerado: gold_indicador_municipio.parquet | linhas=23995 | colunas=51 | tamanho_mb=1.0121


,ano,id_municipio,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,...,media_peso_aluno,source_table_alunos,processed_at_alunos,layer_alunos,ano_meta_referencia,meta_referencia,distancia_meta_pontos,status_meta,distancia_meta_2030_pontos,gold_dataset
0,2023,1100031,2,3,69.10,767.8763,NaN,NaN,NaN,NaN,...,1.105263,alunos_agregado,2026-07-12 21:01:53,silver,2024,NaN,NaN,sem_meta_referencia,NaN,gold_indicador_municipio
1,2023,1100072,2,3,58.20,747.8918,NaN,NaN,NaN,NaN,...,1.085900,alunos_agregado,2026-07-12 21:01:53,silver,2024,NaN,NaN,sem_meta_referencia,NaN,gold_indicador_municipio
2,2023,1100189,2,5,69.73,762.4062,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2024,NaN,NaN,sem_meta_referencia,NaN,gold_indicador_municipio
3,2023,1101609,2,3,50.70,745.6802,NaN,NaN,NaN,NaN,...,1.056881,alunos_agregado,2026-07-12 21:01:53,silver,2024,NaN,NaN,sem_meta_referencia,NaN,gold_indicador_municipio
4,2023,1101807,2,3,55.69,752.3724,NaN,NaN,NaN,NaN,...,1.203576,alunos_agregado,2026-07-12 21:01:53,silver,2024,NaN,NaN,sem_meta_referencia,NaN,gold_indicador_municipio


### Gold: comparativo meta x resultado municipal

In [38]:
# Gold 2: Comparativo entre resultado municipal e metas

selected_columns = [
    "ano",
    "id_municipio",
    "sigla_uf",
    "rede",
    "serie",
    "taxa_alfabetizacao",
    "media_portugues",
    "meta_referencia",
    "ano_meta_referencia",
    "distancia_meta_pontos",
    "status_meta",
    "meta_alfabetizacao_2024",
    "meta_alfabetizacao_2025",
    "meta_alfabetizacao_2026",
    "meta_alfabetizacao_2027",
    "meta_alfabetizacao_2028",
    "meta_alfabetizacao_2029",
    "meta_alfabetizacao_2030",
    "distancia_meta_2030_pontos",
    "percentual_participacao_meta",
    "nivel_alfabetizacao",
    "total_alunos",
    "media_proficiencia",
]

available_columns = [col for col in selected_columns if col in df_gold_indicador_municipio.columns]

df_gold_comparativo_municipio = df_gold_indicador_municipio[available_columns].copy()
df_gold_comparativo_municipio = add_gold_metadata(
    df_gold_comparativo_municipio,
    "gold_comparativo_meta_resultado_municipio"
)

validate_gold_dataset(
    df_gold_comparativo_municipio,
    "gold_comparativo_meta_resultado_municipio",
    ["ano", "id_municipio", "rede", "taxa_alfabetizacao", "status_meta"]
)

save_gold_dataset(
    df_gold_comparativo_municipio,
    "gold_comparativo_meta_resultado_municipio.parquet",
    "Comparativo entre taxa de alfabetização municipal e metas de referência"
)

display(df_gold_comparativo_municipio.head())

2026-07-12 21:01:54 | INFO | Dataset Gold gerado: gold_comparativo_meta_resultado_municipio.parquet | linhas=23995 | colunas=26 | tamanho_mb=0.4164


,ano,id_municipio,sigla_uf,rede,serie,taxa_alfabetizacao,media_portugues,meta_referencia,ano_meta_referencia,distancia_meta_pontos,...,meta_alfabetizacao_2029,meta_alfabetizacao_2030,distancia_meta_2030_pontos,percentual_participacao_meta,nivel_alfabetizacao,total_alunos,media_proficiencia,gold_dataset,processed_at,layer
0,2023,1100031,RO,3,2,69.10,767.8763,NaN,2024,NaN,...,NaN,NaN,NaN,NaN,<NA>,84,767.675528,gold_comparativo_meta_resultado_municipio,2026-07-12 21:01:54,gold
1,2023,1100072,RO,3,2,58.20,747.8918,NaN,2024,NaN,...,NaN,NaN,NaN,NaN,<NA>,129,749.537366,gold_comparativo_meta_resultado_municipio,2026-07-12 21:01:54,gold
2,2023,1100189,RO,5,2,69.73,762.4062,NaN,2024,NaN,...,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,gold_comparativo_meta_resultado_municipio,2026-07-12 21:01:54,gold
3,2023,1101609,RO,3,2,50.70,745.6802,NaN,2024,NaN,...,NaN,NaN,NaN,NaN,<NA>,100,745.688467,gold_comparativo_meta_resultado_municipio,2026-07-12 21:01:54,gold
4,2023,1101807,RO,3,2,55.69,752.3724,NaN,2024,NaN,...,NaN,NaN,NaN,NaN,<NA>,89,758.844213,gold_comparativo_meta_resultado_municipio,2026-07-12 21:01:54,gold


### Gold: ranking de municípios prioritários

In [39]:
# Gold 3: Ranking de municípios prioritários

df_gold_ranking_municipios = df_gold_comparativo_municipio.copy()

df_gold_ranking_municipios["taxa_alfabetizacao"] = pd.to_numeric(
    df_gold_ranking_municipios["taxa_alfabetizacao"],
    errors="coerce"
)

df_gold_ranking_municipios["distancia_meta_pontos"] = pd.to_numeric(
    df_gold_ranking_municipios["distancia_meta_pontos"],
    errors="coerce"
)

# Score simples de priorização:
# Quanto mais negativa a distância da meta e menor a taxa de alfabetização,
# maior a prioridade.
df_gold_ranking_municipios["score_prioridade"] = (
    df_gold_ranking_municipios["distancia_meta_pontos"].abs().fillna(0) * 0.7
    + (100 - df_gold_ranking_municipios["taxa_alfabetizacao"]).clip(lower=0).fillna(0) * 0.3
)

df_gold_ranking_municipios = df_gold_ranking_municipios[
    df_gold_ranking_municipios["status_meta"] == "abaixo_da_meta"
].copy()

df_gold_ranking_municipios = df_gold_ranking_municipios.sort_values(
    by=["ano", "rede", "score_prioridade"],
    ascending=[True, True, False]
)

df_gold_ranking_municipios["ranking_prioridade"] = (
    df_gold_ranking_municipios
    .groupby(["ano", "rede"])["score_prioridade"]
    .rank(method="dense", ascending=False)
    .astype("Int64")
)

df_gold_ranking_municipios = add_gold_metadata(
    df_gold_ranking_municipios,
    "gold_ranking_municipios_prioritarios"
)

validate_gold_dataset(
    df_gold_ranking_municipios,
    "gold_ranking_municipios_prioritarios",
    ["ano", "id_municipio", "rede", "taxa_alfabetizacao", "score_prioridade", "ranking_prioridade"]
)

save_gold_dataset(
    df_gold_ranking_municipios,
    "gold_ranking_municipios_prioritarios.parquet",
    "Ranking de municípios abaixo da meta com maior prioridade de intervenção"
)

display(df_gold_ranking_municipios.head(20))

2026-07-12 21:01:54 | INFO | Dataset Gold gerado: gold_ranking_municipios_prioritarios.parquet | linhas=0 | colunas=28 | tamanho_mb=0.0135


,ano,id_municipio,sigla_uf,rede,serie,taxa_alfabetizacao,media_portugues,meta_referencia,ano_meta_referencia,distancia_meta_pontos,...,distancia_meta_2030_pontos,percentual_participacao_meta,nivel_alfabetizacao,total_alunos,media_proficiencia,gold_dataset,processed_at,layer,score_prioridade,ranking_prioridade


### Gold: indicador por UF

In [40]:
# Gold 4: Indicador de alfabetização por UF

df_meta_uf_join = df_meta_uf.copy()

df_meta_uf_join = df_meta_uf_join.rename(
    columns={
        "taxa_alfabetizacao": "taxa_alfabetizacao_meta_base",
        "percentual_participacao": "percentual_participacao_meta"
    }
)

df_gold_indicador_uf = df_indicador_uf.merge(
    df_meta_uf_join,
    on=["ano", "sigla_uf", "rede"],
    how="left",
    suffixes=("", "_meta")
)

df_gold_indicador_uf = add_meta_reference(df_gold_indicador_uf)
df_gold_indicador_uf = add_gold_metadata(
    df_gold_indicador_uf,
    "gold_indicador_uf"
)

validate_gold_dataset(
    df_gold_indicador_uf,
    "gold_indicador_uf",
    ["ano", "sigla_uf", "rede", "serie", "taxa_alfabetizacao"]
)

save_gold_dataset(
    df_gold_indicador_uf,
    "gold_indicador_uf.parquet",
    "Indicador de alfabetização por UF integrado às metas estaduais"
)

display(df_gold_indicador_uf.head())

2026-07-12 21:01:54 | INFO | Dataset Gold gerado: gold_indicador_uf.parquet | linhas=145 | colunas=36 | tamanho_mb=0.0269


,ano,sigla_uf,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,...,percentual_participacao_meta,source_table_meta,processed_at_meta,layer_meta,ano_meta_referencia,meta_referencia,distancia_meta_pontos,status_meta,distancia_meta_2030_pontos,gold_dataset
0,2023,AM,2,3,49.20,733.6637,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2024,NaN,NaN,sem_meta_referencia,NaN,gold_indicador_uf
1,2023,PB,2,2,55.23,744.8152,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2024,NaN,NaN,sem_meta_referencia,NaN,gold_indicador_uf
2,2023,PR,2,5,73.12,757.2146,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2024,NaN,NaN,sem_meta_referencia,NaN,gold_indicador_uf
3,2023,AP,2,3,41.87,732.7858,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2024,NaN,NaN,sem_meta_referencia,NaN,gold_indicador_uf
4,2023,PE,2,5,58.95,747.4522,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2024,NaN,NaN,sem_meta_referencia,NaN,gold_indicador_uf


### Gold: evolução temporal

In [41]:
# Gold 5: Evolução temporal da alfabetização por UF e rede

df_gold_evolucao = df_gold_indicador_uf.copy()

evolucao_columns = [
    "ano",
    "sigla_uf",
    "rede",
    "serie",
    "taxa_alfabetizacao",
    "media_portugues",
    "meta_referencia",
    "distancia_meta_pontos",
    "status_meta"
]

evolucao_columns = [col for col in evolucao_columns if col in df_gold_evolucao.columns]

df_gold_evolucao = df_gold_evolucao[evolucao_columns].copy()

df_gold_evolucao = df_gold_evolucao.sort_values(
    by=["sigla_uf", "rede", "serie", "ano"]
)

df_gold_evolucao["variacao_taxa_alfabetizacao"] = (
    df_gold_evolucao
    .groupby(["sigla_uf", "rede", "serie"])["taxa_alfabetizacao"]
    .diff()
)

df_gold_evolucao = add_gold_metadata(
    df_gold_evolucao,
    "gold_evolucao_alfabetizacao_uf"
)

validate_gold_dataset(
    df_gold_evolucao,
    "gold_evolucao_alfabetizacao_uf",
    ["ano", "sigla_uf", "rede", "taxa_alfabetizacao"]
)

save_gold_dataset(
    df_gold_evolucao,
    "gold_evolucao_alfabetizacao_uf.parquet",
    "Evolução temporal da taxa de alfabetização por UF e rede"
)

display(df_gold_evolucao.head(20))

2026-07-12 21:01:54 | INFO | Dataset Gold gerado: gold_evolucao_alfabetizacao_uf.parquet | linhas=145 | colunas=13 | tamanho_mb=0.0102


,ano,sigla_uf,rede,serie,taxa_alfabetizacao,media_portugues,meta_referencia,distancia_meta_pontos,status_meta,variacao_taxa_alfabetizacao,gold_dataset,processed_at,layer
125,2024,AC,2,2,55.50,743.1700,NaN,NaN,sem_meta_referencia,NaN,gold_evolucao_alfabetizacao_uf,2026-07-12 21:01:54,gold
140,2024,AC,3,2,48.16,736.0100,NaN,NaN,sem_meta_referencia,NaN,gold_evolucao_alfabetizacao_uf,2026-07-12 21:01:54,gold
135,2024,AC,5,2,51.38,739.1500,NaN,NaN,sem_meta_referencia,NaN,gold_evolucao_alfabetizacao_uf,2026-07-12 21:01:54,gold
22,2023,AL,2,2,38.65,724.7993,NaN,NaN,sem_meta_referencia,NaN,gold_evolucao_alfabetizacao_uf,2026-07-12 21:01:54,gold
141,2024,AL,2,2,39.92,726.3900,NaN,NaN,sem_meta_referencia,1.27,gold_evolucao_alfabetizacao_uf,2026-07-12 21:01:54,gold
68,2023,AL,3,2,44.13,729.9581,NaN,NaN,sem_meta_referencia,NaN,gold_evolucao_alfabetizacao_uf,2026-07-12 21:01:54,gold
138,2024,AL,3,2,49.05,735.9400,NaN,NaN,sem_meta_referencia,4.92,gold_evolucao_alfabetizacao_uf,2026-07-12 21:01:54,gold
8,2023,AL,5,2,43.88,729.7227,NaN,NaN,sem_meta_referencia,NaN,gold_evolucao_alfabetizacao_uf,2026-07-12 21:01:54,gold
139,2024,AL,5,2,48.63,735.5100,NaN,NaN,sem_meta_referencia,4.75,gold_evolucao_alfabetizacao_uf,2026-07-12 21:01:54,gold
37,2023,AM,2,2,62.63,746.2058,NaN,NaN,sem_meta_referencia,NaN,gold_evolucao_alfabetizacao_uf,2026-07-12 21:01:54,gold


### Gold: base preparada para IA

In [42]:
# Gold 6: Base preparada para aplicações futuras de IA

ia_columns = [
    "ano",
    "id_municipio",
    "sigla_uf",
    "rede",
    "serie",
    "taxa_alfabetizacao",
    "media_portugues",
    "proporcao_aluno_nivel_0",
    "proporcao_aluno_nivel_1",
    "proporcao_aluno_nivel_2",
    "proporcao_aluno_nivel_3",
    "proporcao_aluno_nivel_4",
    "proporcao_aluno_nivel_5",
    "proporcao_aluno_nivel_6",
    "proporcao_aluno_nivel_7",
    "proporcao_aluno_nivel_8",
    "total_alunos",
    "total_com_proficiencia",
    "media_proficiencia",
    "menor_proficiencia",
    "maior_proficiencia",
    "total_com_status_alfabetizacao",
    "total_com_status_presenca",
    "media_peso_aluno",
    "meta_referencia",
    "distancia_meta_pontos",
    "status_meta",
    "meta_alfabetizacao_2030",
    "distancia_meta_2030_pontos",
    "percentual_participacao_meta",
    "nivel_alfabetizacao",
]

available_ia_columns = [col for col in ia_columns if col in df_gold_indicador_municipio.columns]

df_gold_base_ia = df_gold_indicador_municipio[available_ia_columns].copy()

df_gold_base_ia = add_gold_metadata(
    df_gold_base_ia,
    "gold_base_ia_alfabetizacao"
)

validate_gold_dataset(
    df_gold_base_ia,
    "gold_base_ia_alfabetizacao",
    ["ano", "id_municipio", "sigla_uf", "rede", "taxa_alfabetizacao"]
)

save_gold_dataset(
    df_gold_base_ia,
    "gold_base_ia_alfabetizacao.parquet",
    "Base analítica preparada para futuras aplicações de IA"
)

display(df_gold_base_ia.head())

2026-07-12 21:01:55 | INFO | Dataset Gold gerado: gold_base_ia_alfabetizacao.parquet | linhas=23995 | colunas=34 | tamanho_mb=0.9806


,ano,id_municipio,sigla_uf,rede,serie,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,...,meta_referencia,distancia_meta_pontos,status_meta,meta_alfabetizacao_2030,distancia_meta_2030_pontos,percentual_participacao_meta,nivel_alfabetizacao,gold_dataset,processed_at,layer
0,2023,1100031,RO,3,2,69.10,767.8763,NaN,NaN,NaN,...,NaN,NaN,sem_meta_referencia,NaN,NaN,NaN,<NA>,gold_base_ia_alfabetizacao,2026-07-12 21:01:54,gold
1,2023,1100072,RO,3,2,58.20,747.8918,NaN,NaN,NaN,...,NaN,NaN,sem_meta_referencia,NaN,NaN,NaN,<NA>,gold_base_ia_alfabetizacao,2026-07-12 21:01:54,gold
2,2023,1100189,RO,5,2,69.73,762.4062,NaN,NaN,NaN,...,NaN,NaN,sem_meta_referencia,NaN,NaN,NaN,<NA>,gold_base_ia_alfabetizacao,2026-07-12 21:01:54,gold
3,2023,1101609,RO,3,2,50.70,745.6802,NaN,NaN,NaN,...,NaN,NaN,sem_meta_referencia,NaN,NaN,NaN,<NA>,gold_base_ia_alfabetizacao,2026-07-12 21:01:54,gold
4,2023,1101807,RO,3,2,55.69,752.3724,NaN,NaN,NaN,...,NaN,NaN,sem_meta_referencia,NaN,NaN,NaN,<NA>,gold_base_ia_alfabetizacao,2026-07-12 21:01:54,gold


### Salvar manifesto e qualidade da Gold

In [43]:
# Salvando manifesto e relatório de qualidade da camada Gold

gold_manifest_file = GOLD_OUTPUT_PATH / "gold_transform_manifest.json"
gold_quality_file = GOLD_OUTPUT_PATH / "gold_quality_report.json"

with open(gold_manifest_file, "w", encoding="utf-8") as file:
    json.dump(gold_manifest, file, ensure_ascii=False, indent=4)

with open(gold_quality_file, "w", encoding="utf-8") as file:
    json.dump(gold_quality_report, file, ensure_ascii=False, indent=4)

df_gold_manifest = pd.DataFrame(gold_manifest)
df_gold_quality = pd.DataFrame(gold_quality_report)

display(df_gold_manifest)
display(df_gold_quality)

write_log(f"Manifesto Gold gerado: {gold_manifest_file}")
write_log(f"Relatório de qualidade Gold gerado: {gold_quality_file}")

,description,output_file,rows,columns,file_size_mb,elapsed_seconds,status
0,Indicador de alfabetização por município integ...,/content/techchallenge-fase2-pipeline-alfabeti...,23995,51,1.0121,0.09,success
1,Comparativo entre taxa de alfabetização munici...,/content/techchallenge-fase2-pipeline-alfabeti...,23995,26,0.4164,0.04,success
2,Ranking de municípios abaixo da meta com maior...,/content/techchallenge-fase2-pipeline-alfabeti...,0,28,0.0135,0.00,success
3,Indicador de alfabetização por UF integrado às...,/content/techchallenge-fase2-pipeline-alfabeti...,145,36,0.0269,0.01,success
4,Evolução temporal da taxa de alfabetização por...,/content/techchallenge-fase2-pipeline-alfabeti...,145,13,0.0102,0.00,success
5,Base analítica preparada para futuras aplicaçõ...,/content/techchallenge-fase2-pipeline-alfabeti...,23995,34,0.9806,0.06,success


,dataset_name,validation_type,status,errors
0,gold_indicador_municipio,required_fields,approved,{}
1,gold_comparativo_meta_resultado_municipio,required_fields,approved,{}
2,gold_ranking_municipios_prioritarios,required_fields,approved,{}
3,gold_indicador_uf,required_fields,approved,{}
4,gold_evolucao_alfabetizacao_uf,required_fields,approved,{}
5,gold_base_ia_alfabetizacao,required_fields,approved,{}


2026-07-12 21:01:55 | INFO | Manifesto Gold gerado: /content/techchallenge-fase2-pipeline-alfabetizacao/data/gold/gold_transform_manifest.json
2026-07-12 21:01:55 | INFO | Relatório de qualidade Gold gerado: /content/techchallenge-fase2-pipeline-alfabetizacao/data/gold/gold_quality_report.json


### Validação dos arquivos Gold esperados

In [44]:
# Validação dos arquivos esperados na camada Gold

expected_gold_files = [
    "gold_indicador_municipio.parquet",
    "gold_comparativo_meta_resultado_municipio.parquet",
    "gold_ranking_municipios_prioritarios.parquet",
    "gold_indicador_uf.parquet",
    "gold_evolucao_alfabetizacao_uf.parquet",
    "gold_base_ia_alfabetizacao.parquet",
    "gold_transform_manifest.json",
    "gold_quality_report.json",
]

gold_validation = []

for file_name in expected_gold_files:
    file_path = GOLD_OUTPUT_PATH / file_name

    gold_validation.append({
        "file_name": file_name,
        "exists": file_path.exists(),
        "size_mb": round(file_path.stat().st_size / 1024 / 1024, 4) if file_path.exists() else 0
    })

df_gold_validation = pd.DataFrame(gold_validation)

display(df_gold_validation)

if df_gold_validation["exists"].all():
    write_log("Validação da camada Gold aprovada: todos os arquivos esperados foram gerados.")
else:
    missing = df_gold_validation[df_gold_validation["exists"] == False]["file_name"].tolist()
    write_log(f"Arquivos ausentes na camada Gold: {missing}", level="ERROR")

,file_name,exists,size_mb
0,gold_indicador_municipio.parquet,True,1.0121
1,gold_comparativo_meta_resultado_municipio.parquet,True,0.4164
2,gold_ranking_municipios_prioritarios.parquet,True,0.0135
3,gold_indicador_uf.parquet,True,0.0269
4,gold_evolucao_alfabetizacao_uf.parquet,True,0.0102
5,gold_base_ia_alfabetizacao.parquet,True,0.9806
6,gold_transform_manifest.json,True,0.0021
7,gold_quality_report.json,True,0.0010


2026-07-12 21:01:55 | INFO | Validação da camada Gold aprovada: todos os arquivos esperados foram gerados.


### Resumo da Gold

In [45]:
# Resumo executivo da camada Gold

total_gold_files = len(expected_gold_files)
total_gold_rows = int(df_gold_manifest["rows"].sum())
total_gold_size_mb = round(df_gold_validation["size_mb"].sum(), 4)

gold_summary = {
    "total_files_generated": total_gold_files,
    "total_rows_processed": total_gold_rows,
    "total_size_mb": total_gold_size_mb,
    "status": "approved" if df_gold_validation["exists"].all() else "failed"
}

display(pd.DataFrame([gold_summary]))

write_log(
    f"Resumo Gold: arquivos={total_gold_files}, linhas_processadas={total_gold_rows}, "
    f"tamanho_mb={total_gold_size_mb}, status={gold_summary['status']}"
)

,total_files_generated,total_rows_processed,total_size_mb,status
0,8,72275,2.4628,approved


2026-07-12 21:01:55 | INFO | Resumo Gold: arquivos=8, linhas_processadas=72275, tamanho_mb=2.4628, status=approved


### Listar arquivos Gold gerados

In [46]:
# Listagem dos arquivos gerados na camada Gold

!find /content/techchallenge-fase2-pipeline-alfabetizacao/data/gold -maxdepth 2 -type f

/content/techchallenge-fase2-pipeline-alfabetizacao/data/gold/gold_quality_report.json
/content/techchallenge-fase2-pipeline-alfabetizacao/data/gold/gold_indicador_uf.parquet
/content/techchallenge-fase2-pipeline-alfabetizacao/data/gold/gold_indicador_municipio.parquet
/content/techchallenge-fase2-pipeline-alfabetizacao/data/gold/gold_evolucao_alfabetizacao_uf.parquet
/content/techchallenge-fase2-pipeline-alfabetizacao/data/gold/gold_transform_manifest.json
/content/techchallenge-fase2-pipeline-alfabetizacao/data/gold/gold_ranking_municipios_prioritarios.parquet
/content/techchallenge-fase2-pipeline-alfabetizacao/data/gold/gold_comparativo_meta_resultado_municipio.parquet
/content/techchallenge-fase2-pipeline-alfabetizacao/data/gold/gold_base_ia_alfabetizacao.parquet


## Nesta etapa, simulamos a ingestão de eventos em tempo quase real.

### Como o projeto segue a restrição de custo zero, não utilizamos Pub/Sub, Kafka ou serviços pagos. Em vez disso, implementamos uma simulação com Python, arquivos JSONL e processamento em micro-batches.

### O fluxo simulado é:

Producer Python → Eventos JSONL → Bronze Streaming → Consumer Python → Silver Streaming → Gold Streaming

### Tipos de eventos simulados:

- atualização de indicador;
- nova medição de desempenho;
- atualização de meta ou resultado.

### Também são geradas métricas operacionais de streaming, incluindo volume processado, eventos válidos, eventos rejeitados e latência média.

### Configuração do Streaming

In [47]:
# Configuração do Streaming Simulado

import uuid
import random
from datetime import datetime, timedelta

if "SILVER_OUTPUT_PATH" not in globals():
    SILVER_OUTPUT_PATH = SILVER_PATH

if "GOLD_OUTPUT_PATH" not in globals():
    GOLD_OUTPUT_PATH = GOLD_PATH

STREAMING_RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
STREAMING_EVENT_COUNT = 120
MICRO_BATCH_SIZE = 25

STREAMING_BRONZE_EVENTS_PATH = BRONZE_STREAMING_PATH / "events"
STREAMING_BRONZE_QUARANTINE_PATH = BRONZE_STREAMING_PATH / "quarantine"
STREAMING_SILVER_PATH = SILVER_OUTPUT_PATH / "streaming"
STREAMING_GOLD_PATH = GOLD_OUTPUT_PATH / "streaming"

STREAMING_BRONZE_EVENTS_PATH.mkdir(parents=True, exist_ok=True)
STREAMING_BRONZE_QUARANTINE_PATH.mkdir(parents=True, exist_ok=True)
STREAMING_SILVER_PATH.mkdir(parents=True, exist_ok=True)
STREAMING_GOLD_PATH.mkdir(parents=True, exist_ok=True)

write_log("Passo 9 iniciado: Streaming simulado.")
write_log(f"Streaming run id: {STREAMING_RUN_ID}")
write_log(f"Quantidade de eventos simulados: {STREAMING_EVENT_COUNT}")
write_log(f"Tamanho do micro-batch: {MICRO_BATCH_SIZE}")

2026-07-12 21:01:55 | INFO | Passo 9 iniciado: Streaming simulado.
2026-07-12 21:01:55 | INFO | Streaming run id: 20260712_210155
2026-07-12 21:01:55 | INFO | Quantidade de eventos simulados: 120
2026-07-12 21:01:55 | INFO | Tamanho do micro-batch: 25


### Funções auxiliares do Producer

In [48]:
# Funções auxiliares para geração dos eventos de streaming

def safe_json_value(value):
    """
    Converte valores Pandas/Numpy para tipos compatíveis com JSON.
    """
    if pd.isna(value):
        return None

    if isinstance(value, (np.integer,)):
        return int(value)

    if isinstance(value, (np.floating,)):
        return float(value)

    return value


def load_streaming_reference_base() -> pd.DataFrame:
    """
    Usa a camada Gold municipal como base realista para gerar eventos simulados.
    """
    reference_file = GOLD_OUTPUT_PATH / "gold_comparativo_meta_resultado_municipio.parquet"

    if not reference_file.exists():
        raise FileNotFoundError(
            "Arquivo Gold municipal não encontrado. Execute primeiro o Passo 8."
        )

    df_reference = pd.read_parquet(reference_file)

    required_columns = [
        "ano",
        "id_municipio",
        "sigla_uf",
        "rede",
        "serie",
        "taxa_alfabetizacao",
        "media_portugues",
        "meta_referencia",
        "status_meta",
    ]

    available_columns = [col for col in required_columns if col in df_reference.columns]
    df_reference = df_reference[available_columns].copy()

    df_reference = df_reference.dropna(
        subset=["ano", "id_municipio", "rede", "serie", "taxa_alfabetizacao"]
    )

    write_log(
        f"Base de referência para streaming carregada: "
        f"linhas={len(df_reference)} | colunas={len(df_reference.columns)}"
    )

    return df_reference

### Producer: gerar eventos JSON

In [49]:
# Producer: geração de eventos simulados em quase tempo real

def generate_streaming_events(df_reference: pd.DataFrame, event_count: int) -> list:
    """
    Gera eventos simulados com base nos dados reais da camada Gold.
    """
    event_types = [
        "atualizacao_indicador",
        "nova_medicao_desempenho",
        "atualizacao_meta_resultado",
    ]

    events = []

    sample_df = df_reference.sample(
        n=event_count,
        replace=True,
        random_state=42
    ).reset_index(drop=True)

    for index, row in sample_df.iterrows():
        event_type = random.choice(event_types)

        base_taxa = float(row["taxa_alfabetizacao"])
        delta_taxa = random.uniform(-3.0, 3.0)
        taxa_evento = max(0, min(100, base_taxa + delta_taxa))

        base_media_portugues = safe_json_value(row.get("media_portugues"))
        if base_media_portugues is None:
            media_portugues_evento = None
        else:
            media_portugues_evento = float(base_media_portugues) + random.uniform(-10, 10)

        event = {
            "event_id": str(uuid.uuid4()),
            "event_timestamp": (
                datetime.now() - timedelta(seconds=random.randint(1, 300))
            ).strftime("%Y-%m-%d %H:%M:%S"),
            "event_type": event_type,
            "source_system": "simulador_python_colab",
            "ano": int(row["ano"]),
            "id_municipio": str(row["id_municipio"]).zfill(7),
            "sigla_uf": safe_json_value(row.get("sigla_uf")),
            "rede": safe_json_value(row.get("rede")),
            "serie": safe_json_value(row.get("serie")),
            "taxa_alfabetizacao_evento": round(taxa_evento, 4),
            "media_portugues_evento": round(media_portugues_evento, 4) if media_portugues_evento is not None else None,
            "meta_referencia_evento": safe_json_value(row.get("meta_referencia")),
            "status_meta_anterior": safe_json_value(row.get("status_meta")),
        }

        events.append(event)

    # Eventos propositalmente problemáticos para demonstrar validação de qualidade
    invalid_event_taxa = events[0].copy()
    invalid_event_taxa["event_id"] = str(uuid.uuid4())
    invalid_event_taxa["taxa_alfabetizacao_evento"] = 145.0

    invalid_event_municipio = events[1].copy()
    invalid_event_municipio["event_id"] = str(uuid.uuid4())
    invalid_event_municipio["id_municipio"] = None

    duplicate_event = events[2].copy()

    events.extend([
        invalid_event_taxa,
        invalid_event_municipio,
        duplicate_event,
    ])

    write_log(
        f"Eventos simulados gerados: total={len(events)} "
        f"incluindo eventos inválidos e duplicados para teste de qualidade."
    )

    return events


df_streaming_reference = load_streaming_reference_base()
streaming_events = generate_streaming_events(
    df_reference=df_streaming_reference,
    event_count=STREAMING_EVENT_COUNT
)

print(f"Eventos gerados: {len(streaming_events)}")
streaming_events[:3]

2026-07-12 21:01:55 | INFO | Base de referência para streaming carregada: linhas=23995 | colunas=9
2026-07-12 21:01:55 | INFO | Eventos simulados gerados: total=123 incluindo eventos inválidos e duplicados para teste de qualidade.
Eventos gerados: 123


[{'event_id': '73365d07-0a2a-486a-8e5a-61fe81b9d411',
  'event_timestamp': '2026-07-12 20:57:21',
  'event_type': 'atualizacao_meta_resultado',
  'source_system': 'simulador_python_colab',
  'ano': 2024,
  'id_municipio': '5214705',
  'sigla_uf': 'GO',
  'rede': '5',
  'serie': '2',
  'taxa_alfabetizacao_evento': 98.1658,
  'media_portugues_evento': 794.891,
  'meta_referencia_evento': None,
  'status_meta_anterior': 'sem_meta_referencia'},
 {'event_id': 'b2fd5b7c-41b9-4d68-b003-d99e59c507c5',
  'event_timestamp': '2026-07-12 21:00:35',
  'event_type': 'atualizacao_indicador',
  'source_system': 'simulador_python_colab',
  'ano': 2024,
  'id_municipio': '1300904',
  'sigla_uf': 'AM',
  'rede': '5',
  'serie': '2',
  'taxa_alfabetizacao_evento': 73.2095,
  'media_portugues_evento': 752.4004,
  'meta_referencia_evento': None,
  'status_meta_anterior': 'sem_meta_referencia'},
 {'event_id': '89f36888-dfc1-4215-8212-08685a81a402',
  'event_timestamp': '2026-07-12 21:01:21',
  'event_type': 

### Bronze Streaming: salvar eventos brutos JSONL

In [50]:
# Bronze Streaming: salvamento dos eventos brutos em JSONL

bronze_streaming_file = STREAMING_BRONZE_EVENTS_PATH / f"streaming_events_{STREAMING_RUN_ID}.jsonl"

with open(bronze_streaming_file, "w", encoding="utf-8") as file:
    for event in streaming_events:
        file.write(json.dumps(event, ensure_ascii=False) + "\n")

write_log(f"Arquivo Bronze Streaming gerado: {bronze_streaming_file}")

print(f"Arquivo gerado: {bronze_streaming_file}")

2026-07-12 21:01:55 | INFO | Arquivo Bronze Streaming gerado: /content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/streaming/events/streaming_events_20260712_210155.jsonl
Arquivo gerado: /content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/streaming/events/streaming_events_20260712_210155.jsonl


### Consumer: leitura e validação dos eventos

In [51]:
# Consumer: leitura e validação dos eventos de streaming

ALLOWED_EVENT_TYPES = [
    "atualizacao_indicador",
    "nova_medicao_desempenho",
    "atualizacao_meta_resultado",
]

STREAMING_REQUIRED_FIELDS = [
    "event_id",
    "event_timestamp",
    "event_type",
    "ano",
    "id_municipio",
    "rede",
    "serie",
    "taxa_alfabetizacao_evento",
]


def read_jsonl_to_dataframe(file_path: Path) -> pd.DataFrame:
    """
    Lê arquivo JSONL e converte para DataFrame.
    """
    records = []

    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            records.append(json.loads(line))

    df = pd.DataFrame(records)
    write_log(f"Arquivo JSONL lido pelo consumer: {file_path} | eventos={len(df)}")

    return df


def validate_streaming_row(row, seen_event_ids: set) -> list:
    """
    Valida um evento individual de streaming.
    """
    errors = []

    for field in STREAMING_REQUIRED_FIELDS:
        if field not in row or pd.isna(row[field]) or row[field] == "":
            errors.append(f"campo_obrigatorio_ausente:{field}")

    event_id = row.get("event_id")

    if event_id in seen_event_ids:
        errors.append("evento_duplicado:event_id")
    else:
        seen_event_ids.add(event_id)

    if row.get("event_type") not in ALLOWED_EVENT_TYPES:
        errors.append("event_type_invalido")

    id_municipio = row.get("id_municipio")
    if pd.notna(id_municipio):
        id_municipio_str = str(id_municipio)
        if len(id_municipio_str) != 7:
            errors.append("id_municipio_invalido")

    taxa = row.get("taxa_alfabetizacao_evento")
    try:
        taxa_float = float(taxa)
        if taxa_float < 0 or taxa_float > 100:
            errors.append("taxa_alfabetizacao_fora_faixa_0_100")
    except Exception:
        errors.append("taxa_alfabetizacao_nao_numerica")

    return errors


def validate_streaming_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Valida todo o DataFrame de eventos.
    """
    df = df.copy()
    seen_event_ids = set()

    validation_errors = []

    for _, row in df.iterrows():
        validation_errors.append(validate_streaming_row(row, seen_event_ids))

    df["validation_error_count"] = [len(errors) for errors in validation_errors]
    df["validation_errors"] = [
        "|".join(errors) if errors else ""
        for errors in validation_errors
    ]

    df["is_valid"] = df["validation_error_count"] == 0

    return df

### Processamento em micro-batches

In [52]:
# Processamento dos eventos em micro-batches

df_raw_streaming_events = read_jsonl_to_dataframe(bronze_streaming_file)

processed_batches = []
quarantine_batches = []
streaming_microbatch_manifest = []

start_pipeline_time = time.perf_counter()

for batch_number, start_index in enumerate(range(0, len(df_raw_streaming_events), MICRO_BATCH_SIZE), start=1):
    microbatch_start = time.perf_counter()

    batch_df = df_raw_streaming_events.iloc[start_index:start_index + MICRO_BATCH_SIZE].copy()

    consumed_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    batch_df["consumer_batch_id"] = f"{STREAMING_RUN_ID}_batch_{batch_number}"
    batch_df["consumed_at"] = consumed_at

    batch_df = validate_streaming_dataframe(batch_df)

    batch_df["event_timestamp_dt"] = pd.to_datetime(batch_df["event_timestamp"], errors="coerce")
    batch_df["consumed_at_dt"] = pd.to_datetime(batch_df["consumed_at"], errors="coerce")
    batch_df["latency_seconds"] = (
        batch_df["consumed_at_dt"] - batch_df["event_timestamp_dt"]
    ).dt.total_seconds()

    valid_batch = batch_df[batch_df["is_valid"] == True].copy()
    invalid_batch = batch_df[batch_df["is_valid"] == False].copy()

    processed_batches.append(valid_batch)

    if len(invalid_batch) > 0:
        quarantine_batches.append(invalid_batch)

    elapsed_batch = round(time.perf_counter() - microbatch_start, 4)

    streaming_microbatch_manifest.append({
        "batch_number": batch_number,
        "consumer_batch_id": f"{STREAMING_RUN_ID}_batch_{batch_number}",
        "events_received": int(len(batch_df)),
        "events_valid": int(len(valid_batch)),
        "events_invalid": int(len(invalid_batch)),
        "avg_latency_seconds": round(float(batch_df["latency_seconds"].mean()), 4),
        "max_latency_seconds": round(float(batch_df["latency_seconds"].max()), 4),
        "elapsed_processing_seconds": elapsed_batch,
        "status": "success"
    })

    write_log(
        f"Micro-batch processado: batch={batch_number} | "
        f"recebidos={len(batch_df)} | válidos={len(valid_batch)} | inválidos={len(invalid_batch)}"
    )

total_pipeline_elapsed = round(time.perf_counter() - start_pipeline_time, 4)

df_streaming_valid = pd.concat(processed_batches, ignore_index=True) if processed_batches else pd.DataFrame()
df_streaming_invalid = pd.concat(quarantine_batches, ignore_index=True) if quarantine_batches else pd.DataFrame()

print(f"Eventos válidos: {len(df_streaming_valid)}")
print(f"Eventos inválidos/quarentena: {len(df_streaming_invalid)}")

2026-07-12 21:01:55 | INFO | Arquivo JSONL lido pelo consumer: /content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/streaming/events/streaming_events_20260712_210155.jsonl | eventos=123
2026-07-12 21:01:55 | INFO | Micro-batch processado: batch=1 | recebidos=25 | válidos=25 | inválidos=0
2026-07-12 21:01:55 | INFO | Micro-batch processado: batch=2 | recebidos=25 | válidos=25 | inválidos=0
2026-07-12 21:01:55 | INFO | Micro-batch processado: batch=3 | recebidos=25 | válidos=25 | inválidos=0
2026-07-12 21:01:55 | INFO | Micro-batch processado: batch=4 | recebidos=25 | válidos=25 | inválidos=0
2026-07-12 21:01:55 | INFO | Micro-batch processado: batch=5 | recebidos=23 | válidos=21 | inválidos=2
Eventos válidos: 121
Eventos inválidos/quarentena: 2


### Silver Streaming e Quarentena

In [53]:
# Silver Streaming: salvar eventos válidos tratados
# Quarentena: salvar eventos inválidos para auditoria

numeric_streaming_columns = [
    "ano",
    "taxa_alfabetizacao_evento",
    "media_portugues_evento",
    "meta_referencia_evento",
    "validation_error_count",
    "latency_seconds",
]

for column in numeric_streaming_columns:
    if column in df_streaming_valid.columns:
        df_streaming_valid[column] = pd.to_numeric(df_streaming_valid[column], errors="coerce")

if "id_municipio" in df_streaming_valid.columns:
    df_streaming_valid["id_municipio"] = df_streaming_valid["id_municipio"].astype("string").str.zfill(7)

if "sigla_uf" in df_streaming_valid.columns:
    df_streaming_valid["sigla_uf"] = df_streaming_valid["sigla_uf"].astype("string").str.upper()

df_streaming_valid["processed_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
df_streaming_valid["layer"] = "silver_streaming"

silver_streaming_file = STREAMING_SILVER_PATH / f"silver_streaming_eventos_{STREAMING_RUN_ID}.parquet"
df_streaming_valid.to_parquet(silver_streaming_file, index=False)

write_log(f"Arquivo Silver Streaming gerado: {silver_streaming_file}")

if len(df_streaming_invalid) > 0:
    quarantine_file = STREAMING_BRONZE_QUARANTINE_PATH / f"streaming_quarantine_{STREAMING_RUN_ID}.jsonl"
    df_streaming_invalid.to_json(
        quarantine_file,
        orient="records",
        lines=True,
        force_ascii=False
    )
    write_log(f"Arquivo de quarentena Streaming gerado: {quarantine_file}")
else:
    quarantine_file = None
    write_log("Nenhum evento inválido enviado para quarentena.")

print(f"Silver Streaming: {silver_streaming_file}")
print(f"Quarentena: {quarantine_file}")

2026-07-12 21:01:55 | INFO | Arquivo Silver Streaming gerado: /content/techchallenge-fase2-pipeline-alfabetizacao/data/silver/streaming/silver_streaming_eventos_20260712_210155.parquet
2026-07-12 21:01:55 | INFO | Arquivo de quarentena Streaming gerado: /content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/streaming/quarantine/streaming_quarantine_20260712_210155.jsonl
Silver Streaming: /content/techchallenge-fase2-pipeline-alfabetizacao/data/silver/streaming/silver_streaming_eventos_20260712_210155.parquet
Quarentena: /content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/streaming/quarantine/streaming_quarantine_20260712_210155.jsonl


### Gold Streaming: indicadores recentes

In [54]:
# Gold Streaming: criação de visão analítica dos eventos recentes

df_gold_streaming_eventos = df_streaming_valid.copy()

df_gold_streaming_eventos["distancia_meta_evento_pontos"] = (
    pd.to_numeric(df_gold_streaming_eventos["taxa_alfabetizacao_evento"], errors="coerce")
    - pd.to_numeric(df_gold_streaming_eventos["meta_referencia_evento"], errors="coerce")
)

df_gold_streaming_eventos["status_meta_evento"] = np.select(
    [
        df_gold_streaming_eventos["meta_referencia_evento"].isna(),
        df_gold_streaming_eventos["distancia_meta_evento_pontos"] >= 0,
        df_gold_streaming_eventos["distancia_meta_evento_pontos"] < 0,
    ],
    [
        "sem_meta_referencia",
        "atingiu_ou_superou_meta",
        "abaixo_da_meta",
    ],
    default="indefinido"
)

df_gold_streaming_eventos["alerta_prioridade_streaming"] = np.select(
    [
        df_gold_streaming_eventos["distancia_meta_evento_pontos"] < -10,
        df_gold_streaming_eventos["distancia_meta_evento_pontos"] < 0,
        df_gold_streaming_eventos["distancia_meta_evento_pontos"] >= 0,
    ],
    [
        "critico",
        "atencao",
        "normal",
    ],
    default="indefinido"
)

df_gold_streaming_eventos["gold_dataset"] = "gold_streaming_indicadores_recentes"
df_gold_streaming_eventos["processed_at_gold"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
df_gold_streaming_eventos["layer_gold"] = "gold_streaming"

gold_streaming_eventos_file = STREAMING_GOLD_PATH / f"gold_streaming_indicadores_recentes_{STREAMING_RUN_ID}.parquet"

df_gold_streaming_eventos.to_parquet(gold_streaming_eventos_file, index=False)

write_log(f"Arquivo Gold Streaming gerado: {gold_streaming_eventos_file}")

display(df_gold_streaming_eventos.head())

2026-07-12 21:01:55 | INFO | Arquivo Gold Streaming gerado: /content/techchallenge-fase2-pipeline-alfabetizacao/data/gold/streaming/gold_streaming_indicadores_recentes_20260712_210155.parquet


,event_id,event_timestamp,event_type,source_system,ano,id_municipio,sigla_uf,rede,serie,taxa_alfabetizacao_evento,...,consumed_at_dt,latency_seconds,processed_at,layer,distancia_meta_evento_pontos,status_meta_evento,alerta_prioridade_streaming,gold_dataset,processed_at_gold,layer_gold
0,73365d07-0a2a-486a-8e5a-61fe81b9d411,2026-07-12 20:57:21,atualizacao_meta_resultado,simulador_python_colab,2024,5214705,GO,5,2,98.1658,...,2026-07-12 21:01:55,274.0,2026-07-12 21:01:55,silver_streaming,NaN,sem_meta_referencia,indefinido,gold_streaming_indicadores_recentes,2026-07-12 21:01:55,gold_streaming
1,b2fd5b7c-41b9-4d68-b003-d99e59c507c5,2026-07-12 21:00:35,atualizacao_indicador,simulador_python_colab,2024,1300904,AM,5,2,73.2095,...,2026-07-12 21:01:55,80.0,2026-07-12 21:01:55,silver_streaming,NaN,sem_meta_referencia,indefinido,gold_streaming_indicadores_recentes,2026-07-12 21:01:55,gold_streaming
2,89f36888-dfc1-4215-8212-08685a81a402,2026-07-12 21:01:21,atualizacao_meta_resultado,simulador_python_colab,2023,4300703,RS,2,2,91.4103,...,2026-07-12 21:01:55,34.0,2026-07-12 21:01:55,silver_streaming,NaN,sem_meta_referencia,indefinido,gold_streaming_indicadores_recentes,2026-07-12 21:01:55,gold_streaming
3,098dcf42-7e04-4008-9add-9fcb78b1d9e1,2026-07-12 20:58:58,atualizacao_indicador,simulador_python_colab,2023,2514206,PB,5,2,85.5677,...,2026-07-12 21:01:55,177.0,2026-07-12 21:01:55,silver_streaming,NaN,sem_meta_referencia,indefinido,gold_streaming_indicadores_recentes,2026-07-12 21:01:55,gold_streaming
4,49f9f701-f833-4ae3-a8f4-ab571a7ac887,2026-07-12 20:57:46,nova_medicao_desempenho,simulador_python_colab,2024,3161205,MG,5,2,90.7142,...,2026-07-12 21:01:55,249.0,2026-07-12 21:01:55,silver_streaming,NaN,sem_meta_referencia,indefinido,gold_streaming_indicadores_recentes,2026-07-12 21:01:55,gold_streaming


### Gold Streaming: resumo por UF/rede/evento

In [55]:
# Gold Streaming: resumo analítico dos eventos por UF, rede, série e tipo de evento

df_gold_streaming_resumo = (
    df_gold_streaming_eventos
    .groupby(["ano", "sigla_uf", "rede", "serie", "event_type"], dropna=False)
    .agg(
        total_eventos=("event_id", "count"),
        taxa_alfabetizacao_media_evento=("taxa_alfabetizacao_evento", "mean"),
        media_portugues_media_evento=("media_portugues_evento", "mean"),
        meta_referencia_media_evento=("meta_referencia_evento", "mean"),
        distancia_media_meta_evento=("distancia_meta_evento_pontos", "mean"),
        latencia_media_segundos=("latency_seconds", "mean"),
        latencia_maxima_segundos=("latency_seconds", "max"),
    )
    .reset_index()
)

df_gold_streaming_resumo["processed_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
df_gold_streaming_resumo["gold_dataset"] = "gold_streaming_resumo_eventos"

gold_streaming_resumo_file = STREAMING_GOLD_PATH / f"gold_streaming_resumo_eventos_{STREAMING_RUN_ID}.parquet"

df_gold_streaming_resumo.to_parquet(gold_streaming_resumo_file, index=False)

write_log(f"Arquivo Gold Streaming resumo gerado: {gold_streaming_resumo_file}")

display(df_gold_streaming_resumo.head())

2026-07-12 21:01:55 | INFO | Arquivo Gold Streaming resumo gerado: /content/techchallenge-fase2-pipeline-alfabetizacao/data/gold/streaming/gold_streaming_resumo_eventos_20260712_210155.parquet


,ano,sigla_uf,rede,serie,event_type,total_eventos,taxa_alfabetizacao_media_evento,media_portugues_media_evento,meta_referencia_media_evento,distancia_media_meta_evento,latencia_media_segundos,latencia_maxima_segundos,processed_at,gold_dataset
0,2023,AM,2,2,atualizacao_indicador,1,30.74650,728.34340,NaN,NaN,15.0,15.0,2026-07-12 21:01:55,gold_streaming_resumo_eventos
1,2023,BA,2,2,atualizacao_meta_resultado,1,65.59780,752.29940,NaN,NaN,5.0,5.0,2026-07-12 21:01:55,gold_streaming_resumo_eventos
2,2023,BA,3,2,atualizacao_indicador,1,42.27110,723.65500,NaN,NaN,224.0,224.0,2026-07-12 21:01:55,gold_streaming_resumo_eventos
3,2023,BA,3,2,atualizacao_meta_resultado,2,33.66115,717.33165,NaN,NaN,141.5,148.0,2026-07-12 21:01:55,gold_streaming_resumo_eventos
4,2023,BA,3,2,nova_medicao_desempenho,1,49.65890,740.97150,NaN,NaN,133.0,133.0,2026-07-12 21:01:55,gold_streaming_resumo_eventos


### Manifesto e métricas de monitoramento do Streaming

In [56]:
# Manifesto e métricas operacionais do Streaming

df_microbatch_manifest = pd.DataFrame(streaming_microbatch_manifest)

streaming_manifest = {
    "streaming_run_id": STREAMING_RUN_ID,
    "execution_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "producer": "python_colab_jsonl",
    "consumer": "python_colab_microbatch",
    "bronze_file": str(bronze_streaming_file),
    "silver_file": str(silver_streaming_file),
    "gold_eventos_file": str(gold_streaming_eventos_file),
    "gold_resumo_file": str(gold_streaming_resumo_file),
    "quarantine_file": str(quarantine_file) if quarantine_file else None,
    "total_events_generated": int(len(df_raw_streaming_events)),
    "total_events_valid": int(len(df_streaming_valid)),
    "total_events_invalid": int(len(df_streaming_invalid)),
    "micro_batch_size": MICRO_BATCH_SIZE,
    "total_micro_batches": int(len(df_microbatch_manifest)),
    "pipeline_elapsed_seconds": total_pipeline_elapsed,
    "status": "success",
}

streaming_monitoring = {
    "streaming_run_id": STREAMING_RUN_ID,
    "total_events_received": int(len(df_raw_streaming_events)),
    "total_events_valid": int(len(df_streaming_valid)),
    "total_events_invalid": int(len(df_streaming_invalid)),
    "invalid_event_rate": round(len(df_streaming_invalid) / len(df_raw_streaming_events), 4),
    "avg_latency_seconds": round(float(df_streaming_valid["latency_seconds"].mean()), 4),
    "max_latency_seconds": round(float(df_streaming_valid["latency_seconds"].max()), 4),
    "total_micro_batches": int(len(df_microbatch_manifest)),
    "pipeline_elapsed_seconds": total_pipeline_elapsed,
    "alert_status": "attention" if len(df_streaming_invalid) > 0 else "normal",
}

streaming_manifest_file = STREAMING_GOLD_PATH / f"streaming_manifest_{STREAMING_RUN_ID}.json"
streaming_monitoring_file = STREAMING_GOLD_PATH / f"streaming_monitoring_{STREAMING_RUN_ID}.json"
streaming_microbatch_file = STREAMING_GOLD_PATH / f"streaming_microbatch_manifest_{STREAMING_RUN_ID}.json"

with open(streaming_manifest_file, "w", encoding="utf-8") as file:
    json.dump(streaming_manifest, file, ensure_ascii=False, indent=4)

with open(streaming_monitoring_file, "w", encoding="utf-8") as file:
    json.dump(streaming_monitoring, file, ensure_ascii=False, indent=4)

with open(streaming_microbatch_file, "w", encoding="utf-8") as file:
    json.dump(streaming_microbatch_manifest, file, ensure_ascii=False, indent=4)

write_log(f"Manifesto Streaming gerado: {streaming_manifest_file}")
write_log(f"Métricas de monitoramento Streaming geradas: {streaming_monitoring_file}")
write_log(f"Manifesto dos micro-batches gerado: {streaming_microbatch_file}")

display(pd.DataFrame([streaming_monitoring]))
display(df_microbatch_manifest)

2026-07-12 21:01:55 | INFO | Manifesto Streaming gerado: /content/techchallenge-fase2-pipeline-alfabetizacao/data/gold/streaming/streaming_manifest_20260712_210155.json
2026-07-12 21:01:55 | INFO | Métricas de monitoramento Streaming geradas: /content/techchallenge-fase2-pipeline-alfabetizacao/data/gold/streaming/streaming_monitoring_20260712_210155.json
2026-07-12 21:01:55 | INFO | Manifesto dos micro-batches gerado: /content/techchallenge-fase2-pipeline-alfabetizacao/data/gold/streaming/streaming_microbatch_manifest_20260712_210155.json


,streaming_run_id,total_events_received,total_events_valid,total_events_invalid,invalid_event_rate,avg_latency_seconds,max_latency_seconds,total_micro_batches,pipeline_elapsed_seconds,alert_status
0,20260712_210155,123,121,2,0.0163,152.3554,299.0,5,0.0488,attention


,batch_number,consumer_batch_id,events_received,events_valid,events_invalid,avg_latency_seconds,max_latency_seconds,elapsed_processing_seconds,status
0,1,20260712_210155_batch_1,25,25,0,152.8800,299.0,0.0111,success
1,2,20260712_210155_batch_2,25,25,0,134.8800,298.0,0.0085,success
2,3,20260712_210155_batch_3,25,25,0,169.6800,277.0,0.0084,success
3,4,20260712_210155_batch_4,25,25,0,145.4000,271.0,0.0082,success
4,5,20260712_210155_batch_5,23,21,2,161.6522,278.0,0.0094,success


### Validação dos arquivos Streaming

In [57]:
# Validação dos arquivos esperados do Streaming Simulado

expected_streaming_files = [
    bronze_streaming_file,
    silver_streaming_file,
    gold_streaming_eventos_file,
    gold_streaming_resumo_file,
    streaming_manifest_file,
    streaming_monitoring_file,
    streaming_microbatch_file,
]

if quarantine_file:
    expected_streaming_files.append(quarantine_file)

streaming_validation = []

for file_path in expected_streaming_files:
    streaming_validation.append({
        "file_name": file_path.name,
        "path": str(file_path),
        "exists": file_path.exists(),
        "size_mb": round(file_path.stat().st_size / 1024 / 1024, 4) if file_path.exists() else 0
    })

df_streaming_validation = pd.DataFrame(streaming_validation)

display(df_streaming_validation)

if df_streaming_validation["exists"].all():
    write_log("Validação do Streaming aprovada: todos os arquivos esperados foram gerados.")
else:
    missing_streaming = df_streaming_validation[df_streaming_validation["exists"] == False]["file_name"].tolist()
    write_log(f"Arquivos ausentes no Streaming: {missing_streaming}", level="ERROR")

,file_name,path,exists,size_mb
0,streaming_events_20260712_210155.jsonl,/content/techchallenge-fase2-pipeline-alfabeti...,True,0.0489
1,silver_streaming_eventos_20260712_210155.parquet,/content/techchallenge-fase2-pipeline-alfabeti...,True,0.0232
2,gold_streaming_indicadores_recentes_20260712_2...,/content/techchallenge-fase2-pipeline-alfabeti...,True,0.0269
3,gold_streaming_resumo_eventos_20260712_210155....,/content/techchallenge-fase2-pipeline-alfabeti...,True,0.0115
4,streaming_manifest_20260712_210155.json,/content/techchallenge-fase2-pipeline-alfabeti...,True,0.0011
5,streaming_monitoring_20260712_210155.json,/content/techchallenge-fase2-pipeline-alfabeti...,True,0.0003
6,streaming_microbatch_manifest_20260712_210155....,/content/techchallenge-fase2-pipeline-alfabeti...,True,0.0016
7,streaming_quarantine_20260712_210155.jsonl,/content/techchallenge-fase2-pipeline-alfabeti...,True,0.0013


2026-07-12 21:01:55 | INFO | Validação do Streaming aprovada: todos os arquivos esperados foram gerados.


### Listar arquivos Streaming gerados

In [58]:
# Listagem dos arquivos gerados no Streaming

!find /content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/streaming -maxdepth 3 -type f
!find /content/techchallenge-fase2-pipeline-alfabetizacao/data/silver/streaming -maxdepth 3 -type f
!find /content/techchallenge-fase2-pipeline-alfabetizacao/data/gold/streaming -maxdepth 3 -type f

/content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/streaming/events/streaming_events_20260712_210155.jsonl
/content/techchallenge-fase2-pipeline-alfabetizacao/data/bronze/streaming/quarantine/streaming_quarantine_20260712_210155.jsonl
/content/techchallenge-fase2-pipeline-alfabetizacao/data/silver/streaming/silver_streaming_eventos_20260712_210155.parquet
/content/techchallenge-fase2-pipeline-alfabetizacao/data/gold/streaming/streaming_microbatch_manifest_20260712_210155.json
/content/techchallenge-fase2-pipeline-alfabetizacao/data/gold/streaming/streaming_monitoring_20260712_210155.json
/content/techchallenge-fase2-pipeline-alfabetizacao/data/gold/streaming/gold_streaming_indicadores_recentes_20260712_210155.parquet
/content/techchallenge-fase2-pipeline-alfabetizacao/data/gold/streaming/gold_streaming_resumo_eventos_20260712_210155.parquet
/content/techchallenge-fase2-pipeline-alfabetizacao/data/gold/streaming/streaming_manifest_20260712_210155.json
